# GameTheory 3e : Meta-Actions Tarifees et Parcours Complet -- l'agent qui joue, change les regles, puis paie le trajet

[← GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [↑ README GameTheory](README.md) | [jumeau Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) · [jumeau Rust](GameTheory-03e-Meta-Actions-Tarifees-Rust.ipynb)

Les notebooks [GT-3](GameTheory-03-Topology2x2-Python.ipynb), [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) et [GT-3d](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) ont fait de l'espace des jeux 2x2 ordinaux un **univers fini manipulable** : 576 chambres strictes, six swaps generateurs, des murs ou vivent les egalites. Ce notebook monte l'echelle d'un cran : les swaps y deviennent des **actions que l'agent paie**.

## Prerequis- [03-Topology2x2](GameTheory-03-Topology2x2-Python.ipynb) — l'univers fini des 576 jeux 2x2 ordinaux## Version C#

> **Version C# de ce carnet** -- jumeau de [GameTheory-03e-Meta-Actions-Tarifees-Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb), la source pedagogique de reference, et de sa version [Rust](GameTheory-03e-Meta-Actions-Tarifees-Rust.ipynb). Le portage est **pur BCL** : `System`, `System.Collections.Generic`, `System.Linq`, `System.Text.RegularExpressions`, `System.IO` -- **aucune dependance, aucun paquet NuGet, aucun acces reseau**. Les cellules de code sont traduites une a une, dans le meme ordre, et leurs sorties reproduisent celles du carnet Python ligne a ligne : la parite se verifie mecaniquement.

**Correspondance stdlib Python -> BCL** :

| Source Python | Equivalent C# |
|---|---|
| `itertools.product` | requetes LINQ (`from ... from ...`) sur `Enumerable.Range` |
| `collections.Counter` | `Dictionary<K, int>` et l'increment garde `Bump` (en C#, `d[cle] += 1` leve sur une cle absente) |
| `collections.deque` | `Queue<T>` |
| `heapq` (Dijkstra) | `PriorityQueue<TElement, TPriority>`, priorite `(distance, jeu)` : meme ordre de sortie que le tas de tuples de Python |
| `random.seed(0)` / `random.choice` | port BCL du MT19937 de CPython (`init_by_array`, tempering, rejet de `_randbelow`), verifie tirage par tirage ; `System.Random` ne suit pas cet algorithme |
| `pathlib` | `System.IO` : `DirectoryInfo`, `Path.GetRelativePath` |
| `re` | `System.Text.RegularExpressions` |
| `print` de tuples, listes et dictionnaires | helpers `PyTup`, `PyJeu`, `PyLst`, `PyDictInt`, `PyDictIntStr`, qui reproduisent le `repr` de Python |

Les nombres sont formates avec `FormattableString.Invariant` (helper `Fi`) : la locale de la machine ne doit changer aucun caractere de la sortie.

## Plan

1. **Marche 1 -- l'agent joue** : equilibres de Nash purs et dynamique de meilleure reponse sur les 576 jeux
2. **Marche 2 -- l'agent deplace le jeu** : meta-actions tarifees, un cout mesure en echelons de rang, seuil de migration
3. **Marche 3 -- deux agents en desaccord** : le meta-jeu 4x4, ses equilibres, et la question de l'accord

## La question

La strate 6 de la theorie des jeux dit ce qu'un agent fait *dans* des regles donnees. La strate 7 demande ce qui se passe quand **modifier les regles est lui-meme une action** -- avec un cout, donc un arbitrage, donc un equilibre au niveau superieur. Rendu operationnel sur un univers fini, le franchissement devient mesurable : *a quel prix un agent accepte-t-il de deplacer son propre jeu, et que devient cet arbitrage quand les deux joueurs le jouent simultanement ?*

> **Fusion** : ce notebook absorbe l'ex-GameTheory-03f-Parcours-Complet — la partie « parcours complet » (sections 4-8) ci-dessous. L'arc méta-actions (marches 1-3) puis l'intégration (chemin, murs, coûts) se lisent d'un seul tenant.

## 0. Le substrat, herite de GT-3b

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** `R1, R2, R3` (cote Ligne) et `C1, C2, C3` (cote Colonne) echangent les cases portant deux rangs adjacents `k` et `k+1` : ce sont les six generateurs de l'espace, chacun traverse un mur du monde de Bruns-Kimmich. Toute la mecanique est reprise tel quel de [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb), sans modification -- ce notebook est un consommateur du substrat, pas une refonte.

Cote C#, le substrat se reconstruit avec deux alias de tuples de valeur -- `Tab` pour une table de rangs, `Jeu` pour une paire de tables -- qui portent l'egalite structurelle, le hachage et l'ordre lexicographique dont dependent les dictionnaires et le tas de Dijkstra. Le reste est du BCL : `Dictionary`, `HashSet`, `Queue`, `PriorityQueue`, et LINQ pour les produits cartesiens.


In [1]:
// Substrat GT-3b, repris tel quel -- pur BCL (aucun paquet NuGet)
// System, System.Collections.Generic, System.Linq et FormattableString.Invariant suffisent.
using System;
using System.Collections.Generic;
using System.Linq;

// Une table de rangs = un 4-uplet (haut-gauche, haut-droite, bas-gauche, bas-droite), rang 4 = meilleur.
// Un jeu = deux tables. Le substrat est un tuple de VALEUR : egalite structurelle, hachage et
// ordre lexicographique gratuits -- c'est ce qui remplace les tuples Python comme cles de
// dictionnaire et elements du tas de Dijkstra.
using Tab = (int, int, int, int);
using Jeu = ((int, int, int, int), (int, int, int, int));

// --- Helpers de representation : le stdout doit etre identique au carnet Python, ligne a ligne ---
string PyTup(Tab t) => FormattableString.Invariant($"({t.Item1}, {t.Item2}, {t.Item3}, {t.Item4})");
string PyJeu(Jeu g) => $"({PyTup(g.Item1)}, {PyTup(g.Item2)})";
string PyBool(bool b) => b ? "True" : "False";
string PyStr(string s) => "'" + s + "'";
string PyLst<T>(IEnumerable<T> xs, Func<T, string> f) => "[" + string.Join(", ", xs.Select(f)) + "]";
string PyPairIntStr((int N, string Etat) k) => $"({k.N}, {PyStr(k.Etat)})";
string PyDictInt(IEnumerable<KeyValuePair<int, int>> d) =>
    "{" + string.Join(", ", d.OrderBy(kv => kv.Key).Select(kv => FormattableString.Invariant($"{kv.Key}: {kv.Value}"))) + "}";
string PyDictIntStr(IEnumerable<KeyValuePair<(int, string), int>> d) =>
    "{" + string.Join(", ", d.OrderBy(kv => kv.Key).Select(kv => $"{PyPairIntStr(kv.Key)}: {kv.Value}")) + "}";

// Formatage invariant (le helper Fi) : la locale de la machine ne doit changer aucun caractere.
string Fi(FormattableString fs) => FormattableString.Invariant(fs);

// Acces par indice a un 4-uplet, lecture et ecriture des compteurs façon collections.Counter.
int At(Tab t, int i) => i switch { 0 => t.Item1, 1 => t.Item2, 2 => t.Item3, _ => t.Item4 };
int DistinctCount(Tab t) => new HashSet<int> { t.Item1, t.Item2, t.Item3, t.Item4 }.Count;
void Bump<TKey>(IDictionary<TKey, int> d, TKey key) => d[key] = d.TryGetValue(key, out var v) ? v + 1 : 1;

// --- Le substrat proprement dit ---
Tab SwapVal(Tab t, int k)
{
    // Echange les cases portant les rangs k et k+1 (traversee de la facette k<->k+1).
    var l = new[] { t.Item1, t.Item2, t.Item3, t.Item4 };
    int pk = Array.IndexOf(l, k), pk1 = Array.IndexOf(l, k + 1);
    (l[pk], l[pk1]) = (l[pk1], l[pk]);
    return (l[0], l[1], l[2], l[3]);
}

// Le produit cartesien de itertools.product(range(1, 5), repeat=4), filtre sur les permutations.
var chambres = (from a in Enumerable.Range(1, 4)
                from b in Enumerable.Range(1, 4)
                from c in Enumerable.Range(1, 4)
                from d in Enumerable.Range(1, 4)
                where new[] { a, b, c, d }.Distinct().Count() == 4
                orderby a, b, c, d
                select (a, b, c, d)).ToList();
var jeux = (from r in chambres from c in chambres select (r, c)).ToList();
Console.WriteLine($"Chambres strictes par joueur : {chambres.Count} | jeux 2x2 stricts : {jeux.Count}");
Console.WriteLine("Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)");

// Le Dilemme du Prisonnier, encodage GT-21 / GT-3b
var PD = ((3, 1, 4, 2), (3, 4, 1, 2));
Console.WriteLine($"PD : Ligne {PyTup(PD.Item1)} Colonne {PyTup(PD.Item2)} (rang 4 = meilleur, cases TG TD BG BD)");

Chambres strictes par joueur : 24 | jeux 2x2 stricts : 576


Generateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)


PD : Ligne (3, 1, 4, 2) Colonne (3, 4, 1, 2) (rang 4 = meilleur, cases TG TD BG BD)


## 1. Marche 1 -- l'agent joue

Avant de changer les regles, il faut les subir. Pour chaque jeu, deux instruments :

- les **equilibres de Nash purs** : cases ou personne ne prefere devier unilateralement ;
- la **dynamique de meilleure reponse** depuis la case haut-gauche : Ligne ajuste, puis Colonne, et on repete -- c'est la facon la plus simple dont des agents "jouent" quand personne ne calcule d'equilibre.

Sur des rangs *ordinaux*, l'equilibre mixte n'a pas de sens (on ne peut pas moyenner des rangs) : un jeu sans equilibre pur est ici **injouable** -- la dynamique cycle sans fin. C'est un fait, pas une anomalie a corriger.

In [2]:
// Marche 1 : NE purs + dynamique BR depuis la case haut-gauche
List<(int R, int C)> NePurs(Tab row, Tab col)
{
    var nes = new List<(int, int)>();
    foreach (int r in new[] { 0, 1 })
        foreach (int c in new[] { 0, 1 })
            if (At(row, 2 * r + c) > At(row, 2 * (1 - r) + c) && At(col, 2 * r + c) > At(col, 2 * r + (1 - c)))
                nes.Add((r, c));
    return nes;
}

(string Etat, (int R, int C) Case) BrDyn(Tab row, Tab col, (int R, int C) start = default, int maxSteps = 16)
{
    // BR alterne (Ligne d'abord). ("cycle"|"ok", case finale).
    var (r, c) = start;
    for (int step = 0; step < maxSteps; step++)
    {
        bool moved = false;
        int nr = At(row, 2 * r + c) > At(row, 2 * (1 - r) + c) ? r : 1 - r;
        if (nr != r) { r = nr; moved = true; }
        int nc = At(col, 2 * r + c) > At(col, 2 * r + (1 - c)) ? c : 1 - c;
        if (nc != c) { c = nc; moved = true; }
        if (!moved) return ("ok", (r, c));
    }
    return ("cycle", (r, c));
}

var distNe4 = new Dictionary<int, int>();
var stats4 = new Dictionary<string, int>();
var croisement4 = new Dictionary<(int, string), int>();
foreach (var (r4, c4) in jeux)
{
    Bump(distNe4, NePurs(r4, c4).Count);
    Bump(stats4, BrDyn(r4, c4).Etat);
    Bump(croisement4, (NePurs(r4, c4).Count, BrDyn(r4, c4).Etat));
}
Console.WriteLine($"NE purs par jeu : {PyDictInt(distNe4)} sur {jeux.Count}");
Console.WriteLine($"Dynamique BR depuis TG : converge {stats4.GetValueOrDefault("ok")} | cycle {stats4.GetValueOrDefault("cycle")}");
Console.WriteLine("Croisement (#NE, BR) : " + PyDictIntStr(croisement4));
Console.WriteLine();
var nePd4 = PyLst(NePurs(PD.Item1, PD.Item2), n => $"({n.Item1}, {n.Item2})");
Console.WriteLine($"PD : NE purs = {nePd4} -> (bas,droite) = defection mutuelle, rang {At(PD.Item1, 3)} chacun");

NE purs par jeu : {0: 72, 1: 432, 2: 72} sur 576


Dynamique BR depuis TG : converge 504 | cycle 72


Croisement (#NE, BR) : {(0, 'cycle'): 72, (1, 'ok'): 432, (2, 'ok'): 72}


PD : NE purs = [(1, 1)] -> (bas,droite) = defection mutuelle, rang 2 chacun


### Lecture de la marche 1

La distribution est remarquablement symetrique : **72 jeux sans equilibre pur, 432 avec un seul, 72 avec deux** (les coordinations). Et le croisement est exact : la dynamique converge **si et seulement si** un equilibre pur existe -- les 72 cycles sont precisement les 72 jeux sans equilibre. Aucun jeu a deux equilibres ne pose ici de probleme de selection *dynamique* : parti de haut-gauche, le jeu se stabilise toujours quelque part.

Le Dilemme du Prisonnier fait ce qu'on attend de lui : equilibre unique a la defection mutuelle, rang 2 chacun -- le rang 4 de la cooperation existe mais n'est pas stable. Retenons ce chiffre, **2** : c'est ce que vaut le Dilemme pour celui qui y est piege.

In [3]:
// Le fait brut : exhiber un jeu injouable (cycle BR complet)
var exInjouable = ((2, 3, 1, 4), (1, 2, 4, 3));
var (row6, col6) = exInjouable;
Console.WriteLine($"Jeu sans NE pur : Ligne {PyTup(row6)} Colonne {PyTup(col6)}");
var nomsCases6 = new[] { "TG", "TD", "BG", "BD" };
int r6 = 0, c6 = 0;
var trace6 = new List<(int R, int C)> { (0, 0) };
for (int pas6 = 0; pas6 < 6; pas6++)
{
    int nr6 = At(row6, 2 * r6 + c6) > At(row6, 2 * (1 - r6) + c6) ? r6 : 1 - r6;
    if (nr6 != r6) { r6 = nr6; trace6.Add((r6, c6)); }
    int nc6 = At(col6, 2 * r6 + c6) > At(col6, 2 * r6 + (1 - c6)) ? c6 : 1 - c6;
    if (nc6 != c6) { c6 = nc6; trace6.Add((r6, c6)); }
}
Console.WriteLine("Trajectoire BR depuis TG : " + string.Join(" -> ", trace6.Take(6).Select(x => nomsCases6[2 * x.R + x.C])) + " -> ... (cycle)");
Console.WriteLine("Boucle : " + string.Join(" -> ", trace6.Skip(1).Take(4).Select(x => nomsCases6[2 * x.R + x.C])) + " -> ... repetee indefiniment");

// Convention de la suite : un jeu injouable vaut 0 pour chacun (pire que tout rang jouable)
int Payoff(Tab row, Tab col, int cote)
{
    var (st, cas) = BrDyn(row, col);
    if (st == "cycle") return 0;
    return cote == 0 ? At(row, 2 * cas.R + cas.C) : At(col, 2 * cas.R + cas.C);
}
Console.WriteLine();
Console.WriteLine("Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)");

Jeu sans NE pur : Ligne (2, 3, 1, 4) Colonne (1, 2, 4, 3)


Trajectoire BR depuis TG : TG -> TD -> BD -> BG -> TG -> TD -> ... (cycle)


Boucle : TD -> BD -> BG -> TG -> ... repetee indefiniment


Convention : gain au sorti de BR ; jeu injouable -> 0 pour chacun (pire que tout rang >= 1)


## 2. Marche 2 -- l'agent deplace le jeu, et paie

L'agent Ligne recoit desormais des **meta-actions** : appliquer l'un de ses trois swaps `R1, R2, R3` -- reecrire ses propres preference declarees -- puis jouer le jeu deplace. Chaque swap coute **c echelons de rang**. Cette unite n'est pas un artifice : payer 1 signifie *renoncer a un echelon de preference pour l'atteindre*, ce qui rend la soustraction `rang final - cout` honnetement interpretable dans un monde purement ordinal.

Ligne arbitre : rester et toucher son rang courant, ou migrer vers une table distante de d swaps et toucher `rang(final) - c*d`. Comme l'espace est fini, l'optimum se calcule par parcours exhaustif -- BFS depuis **sa propre table** (le graphe de Cayley des 24 tables, diametre 6, deja mesure par GT-3b).

In [4]:
// Marche 2 : optimum de migration par jeu, puis sweep du cout
// BfsFrom rend les distances ET l'ordre de decouverte : l'ordre d'insertion du dictionnaire Python
// porte la selection "premier t a distance d qui atteint p1" de la cellule suivante.
(Dictionary<Tab, int> Dist, List<Tab> Ordre) BfsFrom(Tab t0)
{
    var dist = new Dictionary<Tab, int> { [t0] = 0 };
    var ordre = new List<Tab> { t0 };
    var q = new Queue<Tab>();
    q.Enqueue(t0);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (int k in new[] { 1, 2, 3 })
        {
            var v = SwapVal(u, k);
            if (!dist.ContainsKey(v)) { dist[v] = dist[u] + 1; ordre.Add(v); q.Enqueue(v); }
        }
    }
    return (dist, ordre);
}

var bestByDist = new Dictionary<Jeu, Dictionary<int, int>>();
foreach (var (rowT8, colT8) in jeux)
{
    var m8 = new Dictionary<int, int>();
    var (dist8, ordre8) = BfsFrom(rowT8);
    foreach (var t8 in ordre8)
    {
        int p8 = Payoff(t8, colT8, 0);
        int d8 = dist8[t8];
        m8[d8] = Math.Max(m8.GetValueOrDefault(d8, -1), p8);
    }
    bestByDist[(rowT8, colT8)] = m8;   // m[0] = rang courant, m[d] = meilleur rang atteignable a distance d
}

Console.WriteLine("c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle");
Console.WriteLine(new string('-', 72));
foreach (int c8 in new[] { 0, 1, 2, 3 })
{
    int nMig8 = 0, totD8 = 0, totGain8 = 0, fuit8 = 0;
    foreach (var kv8 in bestByDist)
    {
        var m8 = kv8.Value;
        int best8 = m8.Keys.Max(d => m8[d] - c8 * d);
        int bd8 = m8.Keys.Where(d => m8[d] - c8 * d == best8).Min();
        if (bd8 > 0 && best8 > m8[0])            // migrer ssi strictement meilleur que rester
        {
            nMig8 += 1; totD8 += bd8; totGain8 += best8 - m8[0];
            if (Payoff(kv8.Key.Item1, kv8.Key.Item2, 0) == 0) fuit8 += 1;
        }
    }
    int n8 = jeux.Count;
    Console.WriteLine(Fi($"{c8}   |  {100 * nMig8 / n8,3}%   |     {(double)totD8 / Math.Max(nMig8, 1):F2}     |          {(double)totGain8 / Math.Max(nMig8, 1):+0.00;-0.00;+0.00}          |     {fuit8}"));
}

c   | %migrer | dist.opt moy | gain net moy (migrants) | fuient un cycle


------------------------------------------------------------------------


0   |   56%   |     1.33     |          +1.93          |     72


1   |   16%   |     1.25     |          +2.00          |     72


2   |    8%   |     1.00     |          +1.50          |     48


3   |    4%   |     1.00     |          +1.00          |     24


### Lecture du sweep : un seuil monotone, une distance qui se contracte

Quatre faits mesures :

1. **La migration s'effondre avec le cout** : 56 % des jeux voient Ligne migrer a cout nul, 16 % a cout 1, 8 % a cout 2, 4 % a cout 3. A un echelon par swap, six jeux sur sept restent -- le privilege de reecrire ses preferences est cher.
2. **La distance optimale se contracte** de 1,33 vers 1,00 : plus le km de swap est cher, plus on ne paie que le deplacement minimal -- a cout eleve, seuls les gains de proximite immediate survivent.
3. **Le gain net moyen des migrants monte puis redescend** (+1,93 a c=0, +2,00 a c=1, +1,50 a c=2, +1,00 a c=3) : a c=0 partent aussi les migrations gadgets ; a c=1 ne restent que les migrations de fond ; au-dela, le cout ronge le gain.
4. **Les 72 jeux injouables se vident a leur rythme** : meme a cout 1, les 72 fuient tous (quitter un gain nul vaut n'importe quel prix raisonnable) ; a cout 2 il en reste 48, a cout 3 seulement 24 -- les autres preferent le cycle gratuit au deplacement trop cher. *Meme l'injouable a un prix au-dessus duquel on le tolere.*

In [5]:
// Le Dilemme sous les meta-actions : que peut acheter Ligne ?
var mPd10 = bestByDist[PD];
Console.WriteLine($"PD : rang courant de Ligne (defection mutuelle) = {mPd10[0]}");
Console.WriteLine("Meilleur rang atteignable par distance :");
foreach (int d10 in mPd10.Keys.OrderBy(x => x))
{
    string ligne10 = Fi($"  a distance {d10} : rang {mPd10[d10]}");
    ligne10 += d10 > 0 ? Fi($"  -> net a c=1 : {mPd10[d10] - d10}") : "   (rester)";
    Console.WriteLine(ligne10);
}
Console.WriteLine();
foreach (int c10 in new[] { 0, 1, 2 })
{
    int best10 = mPd10.Keys.Max(d => mPd10[d] - c10 * d);
    int bd10 = mPd10.Keys.Where(d => mPd10[d] - c10 * d == best10).Min();
    string verdict10 = bd10 == 0 ? "RESTE" : $"MIGRE a distance {bd10}";
    Console.WriteLine($"c = {c10} : net optimal {best10} (courant {mPd10[0]}) -> Ligne {verdict10}");
}

PD : rang courant de Ligne (defection mutuelle) = 2


Meilleur rang atteignable par distance :


  a distance 0 : rang 2   (rester)


  a distance 1 : rang 3  -> net a c=1 : 2


  a distance 2 : rang 4  -> net a c=1 : 2


  a distance 3 : rang 4  -> net a c=1 : 1


... (2 outputs left out)

  a distance 6 : rang 4  -> net a c=1 : -2


c = 0 : net optimal 4 (courant 2) -> Ligne MIGRE a distance 2


c = 1 : net optimal 2 (courant 2) -> Ligne RESTE


c = 2 : net optimal 2 (courant 2) -> Ligne RESTE


### Lecture : le Dilemme exactement indifferent

A cout nul, Ligne quitte le Dilemme : deux swaps l'amennent a un jeu ou sa table lui donne le rang 4, net +2. Mais **a cout 1, l'indifference est exacte** : le rang 3 a distance 1 et le rang 4 a distance 2 donnent tous deux un net de 2 -- strictement egal a rester. Le gain de fuite du Dilemme est **entierement mange par le cout du deplacement**. Ce n'est pas un accident d'arrondi : sur l'echelle des rangs, la fuite solitaire du Dilemme coute precisement ce qu'elle rapporte. Le piege de la strate 6 a une propriete economique de plus : il est *juste assez solide* pour retenir un agent solitaire qui doit payer son deplacement -- ce qui rend d'autant plus interessante la question des deux agents, en marche 3.

In [6]:
// Les migrants a cout 1 : qui part, et la fuite des injouables
var mig12 = new List<(Jeu G, int Bd, int P0, int P1)>();
foreach (var kv12 in bestByDist)
{
    var m12 = kv12.Value;
    int best12 = m12.Keys.Max(d => m12[d] - d);
    int bd12 = m12.Keys.Where(d => m12[d] - d == best12).Min();
    if (bd12 > 0 && best12 > m12[0])
        mig12.Add((kv12.Key, bd12, m12[0], m12[bd12]));
}
Console.WriteLine($"Migrants a c=1 : {mig12.Count} jeux sur {jeux.Count} dont {mig12.Count(x => x.P0 == 0)} refugies (partent d'un rang nul)");
var opp12 = mig12.Where(x => x.P0 > 0).ToList();
var premierOpp12 = opp12[0];
// Le premier t de l'ordre BFS qui realise le rang cible : c'est l'ordre d'insertion Python.
var (distOpp12, ordreOpp12) = BfsFrom(premierOpp12.G.Item1);
var tCible = ordreOpp12.First(t => distOpp12[t] == premierOpp12.Bd && Payoff(t, premierOpp12.G.Item2, 0) == premierOpp12.P1);
Console.WriteLine($"Opportuniste : jeu {PyJeu(premierOpp12.G)} | rang {premierOpp12.P0} -> rang {premierOpp12.P1} en {premierOpp12.Bd} swap (table cible {PyTup(tCible)} )");

// Les variables reprises par la cellule 15 (le mur traverse par ce migrant).
var rowT = premierOpp12.G.Item1;
var colT = premierOpp12.G.Item2;
int bdT = premierOpp12.Bd;
int p0T = premierOpp12.P0;
int p1T = premierOpp12.P1;

var inj12 = jeux.Where(g => NePurs(g.Item1, g.Item2).Count == 0).ToList();
int rep1_12 = inj12.Count(g => bestByDist[g].Keys.Where(d => bestByDist[g][d] > 0).Min() == 1);
Console.WriteLine();
Console.WriteLine($"Jeux injouables : {inj12.Count} | reparables en 1 swap : {rep1_12} | autres : {inj12.Count - rep1_12}");
var (rowI12, colI12) = inj12[0];
var (distI12, ordreI12) = BfsFrom(rowI12);
var jouablesD1_12 = ordreI12.Where(t => distI12[t] == 1 && Payoff(t, colI12, 0) > 0).ToList();
int? meilleur12 = jouablesD1_12.Count > 0 ? jouablesD1_12.Max(t => Payoff(t, colI12, 0)) : (int?)null;
Console.WriteLine($"Exemple {PyJeu((rowI12, colI12))} -> tables jouables a distance 1 : {PyLst(jouablesD1_12, t => PyTup(t))} | meilleur rang {meilleur12}");

Migrants a c=1 : 96 jeux sur 576 dont 72 refugies (partent d'un rang nul)


Opportuniste : jeu ((1, 4, 2, 3), (1, 2, 4, 3)) | rang 2 -> rang 4 en 1 swap (table cible (2, 4, 1, 3) )


Jeux injouables : 72 | reparables en 1 swap : 48 | autres : 24


Exemple ((1, 3, 4, 2), (2, 1, 3, 4)) -> tables jouables a distance 1 : [(1, 2, 4, 3)] | meilleur rang 3


### Lecture : deux raisons de partir

Les migrants a cout 1 ne sont pas une population homogene. Il y a les **opportunistes** -- des jeux jouables ou un seul swap proche decalle l'equilibre vers une case de meilleur rang (l'exemple affiche saute de deux echelons en un swap) -- et les **refugies** : les 72 jeux sans equilibre, dont 48 se reparent a distance 1. Pour un refugie, la meta-action n'est pas du luxe strategique, c'est la condition meme de jouabilite : sans elle, le jeu ne produit aucun resultat -- c'est le refugie affiche plus bas, parti du rang nul. La strate 7 commence deja ici -- *changer les regles pour que le jeu existe* -- avant meme toute consideration d'amelioration.

### Le mur traversé — ce que le migrant coupe (critère de clôture #12207)

Le chantier #12207 se clôt quand un lecteur peut, dans un notebook exécuté : partir d'un jeu nommé, atteindre un autre jeu nommé par un chemin qu'il n'a pas écrit lui-même, **voir le mur qu'il traverse**, et lire le coût de la méta-action qui l'y a mené. Les marches 1-2 ont nommé les jeux, calculé les chemins optimaux et balayé les coûts — mais aucune sortie ne **montrait** le mur. Entre deux chambres strictes adjacentes, l'égalité des deux cases permutées est un mur de codimension 1 : la lecture Bruns-Kimmich de GT-3b, rendue visible ici dans le flot de migration payée.

In [7]:
// Le mur traverse : la meta-action echange les rangs k et k+1 de deux cases. Dans l'espace
// des utilites (representant lineaire u = valeur du rang), la deformation
// u(lambda) = (1-lambda)*u_depart + lambda*u_cible coupe l'hyperplan d'egalite des deux
// cases permutees. De part et d'autre, la table ordinale est CONSTANTE : c'est la chambre.
var positions15 = Enumerable.Range(0, 4).Where(i => At(rowT, i) != At(tCible, i)).ToList();
if (positions15.Count != 2) throw new InvalidOperationException("une meta-action = un swap de deux cases");   // garde : equivalent de l'assert Python
int iA15 = positions15[0], iB15 = positions15[1];
var noms15 = new[] { "TG", "TD", "BG", "BD" };

Tab TableDepuisU(double[] u)
{
    // La table ordinale (rangs 1..4) induite par un profil d'utilites.
    var ordre = Enumerable.Range(0, 4).OrderBy(j => u[j]).ToArray();
    var t = new int[4];
    for (int rang = 0; rang < 4; rang++) t[ordre[rang]] = rang + 1;
    return (t[0], t[1], t[2], t[3]);
}

Console.WriteLine($"Jeu depart (Ligne) : {PyTup(rowT)}  ->  cible : {PyTup(tCible)}  ({bdT} meta-action, rang {p0T} -> {p1T})");
Console.WriteLine($"Cases permutees : {noms15[iA15]} (rang {At(rowT, iA15)}) <-> {noms15[iB15]} (rang {At(rowT, iB15)})");
Console.WriteLine();
Console.WriteLine("lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite");
Console.WriteLine(new string('-', 66));
foreach (double lam15 in new[] { 0.0, 0.25, 0.5, 0.75, 1.0 })
{
    var u15 = Enumerable.Range(0, 4).Select(j => (1 - lam15) * At(rowT, j) + lam15 * At(tCible, j)).ToArray();
    string classe15;
    if (Math.Abs(u15[iA15] - u15[iB15]) < 1e-12)
        classe15 = $"MUR : u_{noms15[iA15]} = u_{noms15[iB15]} -- codimension 1";
    else
    {
        var tOrd15 = TableDepuisU(u15);
        classe15 = tOrd15 == rowT ? "chambre de depart" : (tOrd15 == tCible ? "chambre cible" : PyTup(tOrd15));
    }
    Console.WriteLine(Fi($"  {lam15:F2} | ") + string.Join("  ", u15.Select(x => Fi($"{x,5:F2}"))) + $" | {classe15}");
}
Console.WriteLine();
int rangMin15 = Math.Min(At(rowT, iA15), At(rowT, iB15));
Console.WriteLine($"Le migrant coupe UN mur (l'egalite {noms15[iA15]}/{noms15[iB15]} des rangs {rangMin15} et {rangMin15 + 1})");
Console.WriteLine(Fi($"a lambda = 0.50, et l'a paye c = {bdT} : rang {p0T} -> rang {p1T}, gain net {p1T - p0T - bdT:+0;-0;+0}."));

Jeu depart (Ligne) : (1, 4, 2, 3)  ->  cible : (2, 4, 1, 3)  (1 meta-action, rang 2 -> 4)


Cases permutees : TG (rang 1) <-> BG (rang 2)


lambda |   u_TG    u_TD    u_BG    u_BD  | table ordinale induite


------------------------------------------------------------------


  0.00 |  1.00   4.00   2.00   3.00 | chambre de depart


  0.25 |  1.25   4.00   1.75   3.00 | chambre de depart


  0.50 |  1.50   4.00   1.50   3.00 | MUR : u_TG = u_BG -- codimension 1


  0.75 |  1.75   4.00   1.25   3.00 | chambre cible


  1.00 |  2.00   4.00   1.00   3.00 | chambre cible


Le migrant coupe UN mur (l'egalite TG/BG des rangs 1 et 2)


a lambda = 0.50, et l'a paye c = 1 : rang 2 -> rang 4, gain net +1.


### Lecture : les quatre maillons enchaînés

La sortie ci-dessus est le critère de clôture de #12207 réalisé en un seul flot exécuté : un **jeu nommé** (le premier migrant opportuniste, tables affichées), rejoint par un **chemin qu'il n'a pas écrit** (le BFS optimal de la marche 2), en **traversant un mur visible** — la table ordinale reste celle de la chambre de départ pour tout λ < 0.5, celle de la chambre cible pour tout λ > 0.5, et l'égalité des deux cases permutées à λ = 0.50 exactement est le mur de codimension 1 — et le **coût lu** : c = 1 pour cette traversée, gain net +1 en rang.

Le passage payé se lit désormais `chambre -> mur -> chambre voisine` : la méta-action tarifée achète exactement une traversée de mur, et rien d'autre — un échange de rangs qui ne modifie l'ordre d'aucune autre case ne peut couper plus d'un mur.

## 3. Marche 3 -- deux agents, un meta-jeu

Derniere marche : **Ligne et Colonne disposent simultanement de leurs meta-actions**. Chacun choisit une action dans {rester, R1, R2, R3} x {rester, C1, C2, C3} -- chacun ne reecrit que **sa** table, payer ses propres echelons -- puis le jeu deplace est joue (dynamique BR depuis haut-gauche, convention de marche 1), et chacun touche son rang final moins son cout. Seize profils par jeu de base : un **meta-jeu 4x4** dont on peut chercher les equilibres de Nash purs, exactement comme a la marche 1.

La question du desaccord devient calculable : *existe-t-il un equilibre du meta-jeu, bouge-t-on a l'equilibre, et cet equilibre est-il bon pour les deux ?*

In [8]:
// Marche 3 : le meta-jeu 4x4 et ses equilibres (cout = 1)
(int, int) MetaProfil(Tab depRow, Tab depCol, int a1, int a2, int cout = 1)
{
    var r2 = a1 > 0 ? SwapVal(depRow, a1) : depRow;
    var c2 = a2 > 0 ? SwapVal(depCol, a2) : depCol;
    int p1 = Payoff(r2, c2, 0), p2 = Payoff(r2, c2, 1);
    return (p1 - (a1 > 0 ? cout : 0), p2 - (a2 > 0 ? cout : 0));
}

(List<(int, int)> Nes, Dictionary<(int, int), (int, int)> Profils) MetaNe(Tab depRow, Tab depCol, int cout = 1)
{
    var profils = new Dictionary<(int, int), (int, int)>();
    for (int a1 = 0; a1 < 4; a1++)
        for (int a2 = 0; a2 < 4; a2++)
            profils[(a1, a2)] = MetaProfil(depRow, depCol, a1, a2, cout);
    var nes = new List<(int, int)>();
    for (int a1 = 0; a1 < 4; a1++)
        for (int a2 = 0; a2 < 4; a2++)
        {
            var (u1, u2) = profils[(a1, a2)];
            bool okLigne = true, okCol = true;
            for (int a1b = 0; a1b < 4; a1b++) if (profils[(a1b, a2)].Item1 > u1) okLigne = false;
            for (int a2b = 0; a2b < 4; a2b++) if (profils[(a1, a2b)].Item2 > u2) okCol = false;
            if (okLigne && okCol) nes.Add((a1, a2));
        }
    return (nes, profils);
}

var stNe18 = new Dictionary<int, int>();
int tousRestant18 = 0, ambigu18 = 0, aucunRestant18 = 0;
foreach (var g18 in jeux)
{
    var (nes18, _) = MetaNe(g18.Item1, g18.Item2);
    Bump(stNe18, nes18.Count);
    if (nes18.Count == 0) continue;
    if (nes18.All(n => n == (0, 0))) tousRestant18 += 1;
    else if (nes18.Contains((0, 0))) ambigu18 += 1;
    else aucunRestant18 += 1;
}
Console.WriteLine($"Jeux avec au moins un meta-NE pur : {stNe18.Where(kv => kv.Key > 0).Sum(kv => kv.Value)} / {jeux.Count}");
Console.WriteLine("Distribution du nombre de meta-NE : " + PyDictInt(stNe18));
Console.WriteLine();
Console.WriteLine("Statut du mouvement a l'equilibre :");
Console.WriteLine($"  tous les meta-NE restent sur place : {tousRestant18}");
Console.WriteLine($"  (rester) coexiste avec des NE mobiles : {ambigu18} (selection d'equilibre)");
Console.WriteLine($"  aucun meta-NE ne reste sur place : {aucunRestant18} (bouger est necessaire)");

Jeux avec au moins un meta-NE pur : 572 / 576


Distribution du nombre de meta-NE : {0: 4, 1: 166, 2: 266, 3: 16, 4: 116, 5: 6, 6: 2}


Statut du mouvement a l'equilibre :


  tous les meta-NE restent sur place : 134


  (rester) coexiste avec des NE mobiles : 332 (selection d'equilibre)


  aucun meta-NE ne reste sur place : 106 (bouger est necessaire)


### Lecture : le meta-jeu a presque toujours des equilibres

Premiere surprise : **572 jeux sur 576** ont au moins un equilibre pur au niveau meta -- ajouter des meta-actions *stabilise* plutot qu'elle ne destabilise. Le detail du statut est plus riche :

- **134 jeux** : tous les equilibres restent sur place -- la meta-action existe mais personne n'a interet a s'en servir a l'equilibre (l'immobilisme est stable) ;
- **332 jeux** : rester et bouger coexistent comme equilibres -- c'est le domaine de la *selection d'equilibre* : plusieurs futurs possibles, aucun critere ordinal n'en designe un ;
- **106 jeux** : aucun equilibre sur place -- **bouger est une necessite d'equilibre**. Dans pres d'un jeu sur cinq, l'equilibre de strate 6 n'est plus tenable des que reecrire les regles est possible : l'agent qui refuse de payer reste piege pendant que l'autre reconfigure.

La marche 2 disait *quand un agent solitaire part* ; la marche 3 dit que **des qu'on ouvre la porte aux deux, dans un cinquieme des jeux, partir est force**.

In [9]:
// Le meta-jeu du Dilemme, en entier
var (nesPd20, profPd20) = MetaNe(PD.Item1, PD.Item2);
var noms20 = new[] { "reste", "R1", "R2", "R3" };
var nomsC20 = new[] { "reste", "C1", "C2", "C3" };
Console.WriteLine("Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :");
Console.WriteLine("          " + string.Concat(nomsC20.Select(n => Fi($"{n,10}"))));
for (int a1_20 = 0; a1_20 < 4; a1_20++)
{
    string ligne20 = Fi($"{noms20[a1_20],8}  ");
    for (int a2_20 = 0; a2_20 < 4; a2_20++)
    {
        var (u1_20, u2_20) = profPd20[(a1_20, a2_20)];
        string marque20 = nesPd20.Contains((a1_20, a2_20)) ? " *" : "  ";
        ligne20 += $"({u1_20},{u2_20}){marque20} ";
    }
    Console.WriteLine(ligne20);
}
Console.WriteLine();
Console.WriteLine("* = equilibre de Nash pur du meta-jeu : " + PyLst(nesPd20, x => $"({x.Item1}, {x.Item2})"));

Meta-jeu du PD (Ligne en lignes, Colonne en colonnes) :


               reste        C1        C2        C3


   reste  (2,2) * (4,1)   (2,2) * (2,1)   


      R1  (1,4)   (3,1)   (1,3)   (-1,-1)   


      R2  (2,2) * (3,1)   (2,2) * (2,1)   


      R3  (1,2)   (-1,-1)   (1,2)   (3,3) * 


* = equilibre de Nash pur du meta-jeu : [(0, 0), (0, 2), (2, 0), (2, 2), (3, 3)]


### Lecture : le Dilemme se resout -- a deux, et par equilibre

Le meta-jeu du Dilemme porte **cinq equilibres**. Quatre laissent les deux prisonniers a (2,2) -- le statu quo auquel on pense toujours. Mais le cinquieme, **(R3, C3), paie (3,3)** : chacun reecrit sa propre preference au niveau 3-4, le jeu deplace devient une coordination dont l'equilibre selectionne par la dynamique est la case mutuellement meilleure (rang 4 chacun), et meme apres avoir paye un echelon chacun, il reste **(3,3) > (2,2)**. Trois proprietes remarquables :

1. **L'evasion est unilateralement stable** : si un seul migre, la dynamique retombe sur la defection mutuelle et le migrant a paye pour rien -- les profils (R3, reste) et (reste, C3) du tableau paient (1, 2) et (2, 1) : moins que (3, 3) pour chacun -- personne ne peut profiter du départ de l'autre ;
2. **Elle est symetrique** : chacun ne touche qu'a sa table ; personne ne reecrit les preferences de l'autre ;
3. **Elle n'est pas unique** : quatre autres equilibres restent au (2,2). L'evasion *existe et est stable*, mais rien ne la *selectionne* -- c'est un probleme de focalisation, plus de stabilite.

C'est la reponse mesuree a la question de la marche : *modifier les regles est une action, et dans le Dilemme cette action est un equilibre qui domine le piege -- a condition que les deux la jouent ensemble.*

In [10]:
// L'echec de coordination : quand AUCUN equilibre n'est Pareto-optimal
bool ParetoDominated(Dictionary<(int, int), (int, int)> prof, (int, int) p)
    => prof.Values.Any(v => v.Item1 > p.Item1 && v.Item2 > p.Item2);

int auMoinsUnBon22 = 0, aucunBon22 = 0;
(Jeu G, List<(int, int)> Nes, Dictionary<(int, int), (int, int)> Prof)? exDur22 = null;
foreach (var g22 in jeux)
{
    var (nes22, profils22) = MetaNe(g22.Item1, g22.Item2);
    if (nes22.Count == 0) continue;
    if (nes22.Any(n => !ParetoDominated(profils22, profils22[n]))) auMoinsUnBon22 += 1;
    else
    {
        aucunBon22 += 1;
        if (exDur22 is null) exDur22 = (g22, nes22, profils22);
    }
}
Console.WriteLine($"Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : {auMoinsUnBon22}");
Console.WriteLine($"Jeux ou AUCUN equilibre n'est Pareto-optimal : {aucunBon22} -- l'echec de coordination dur");
var (gDur22, nesDur22, profilsDur22) = exDur22.Value;
Console.WriteLine();
var paieRepr22 = PyLst(nesDur22, n => $"({profilsDur22[n].Item1}, {profilsDur22[n].Item2})");
Console.WriteLine($"Exemple {PyJeu(gDur22)} : les meta-NE paient {paieRepr22}");
var frontiere22 = new List<(int, int)>();
for (int a1_22 = 0; a1_22 < 4; a1_22++)
    for (int a2_22 = 0; a2_22 < 4; a2_22++)
    {
        var v22 = profilsDur22[(a1_22, a2_22)];
        if (!ParetoDominated(profilsDur22, v22)) frontiere22.Add(v22);
    }
var frontRepr22 = PyLst(frontiere22.Take(5), v => $"({v.Item1}, {v.Item2})");
Console.WriteLine($"Frontiere de Pareto du meta-jeu : {frontRepr22} ...");

Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : 568


Jeux ou AUCUN equilibre n'est Pareto-optimal : 4 -- l'echec de coordination dur


Exemple ((1, 3, 2, 4), (3, 4, 2, 1)) : les meta-NE paient [(2, 2), (2, 2), (2, 2), (2, 2)]


Frontiere de Pareto du meta-jeu : [(4, 1), (3, 1), (1, 3), (3, 1), (3, 3)] ...


### Lecture : la limite du decentralise

Dans **568 jeux sur 572**, au moins un equilibre du meta-jeu est Pareto-optimal : en general, le meilleur a deux est stable, le probleme est seulement de *le choisir* parmi les equilibres. Mais **4 jeux realisent l'echec de coordination pur** : tous leurs equilibres paient (2,2) tandis que la frontiere de Pareto contient des profils mutuellement meilleurs -- des regles reecrites dont les deux joueurs profiteraient, et qui ne sont **pas stables** : chacun serait tente d'en devier. L'exhibe le montre chiffres en main : l'accord existe dans la matrice, il n'existe pas dans les incitations.

C'est exactement la jonction annoncee par le chantier : la ou la strate 7 rencontre la theorie des mecanismes. Quand changer les regles est une action decentralisee, l'accord mutuel sur le changement reste un equilibre a produire -- le probleme ne fait que remonter d'un etage. Un protocole d'engagement (le versant commitment de [GT-9b](GameTheory-09b-Commitment-Stackelberg-Python.ipynb)) est le candidat naturel, et l'exercice 3 le fait toucher du doigt.

## Limites et conventions (lues avant d'etendre)

- **Le cout en echelons de rang** est une convention locale, documentee : elle rend `rang - cout` interpretable sans cardinaliser les preferences. Un cout en energie, en temps ou en argent exigerait une echelle cardinale et changerait les seuils -- pas la forme du protocole.
- **La regle de jeu** est la dynamique BR depuis la case haut-gauche. Une autre regle de selection (depuis une autre case, tirage aleatoire, dynamique simultanee) deplace certains equilibres selectionnes ; les comptages d'equilibres *purs* (marche 1) et de meta-NE (marche 3) n'en dependent pas.
- **Le meta-jeu est one-shot et simultane** : pas de repetition, pas d'engagement, pas d'ordre de passage. L'exercice 3 ouvre le sequentiel.
- **Chacun ne reecrit que sa table** : le redesign institutionnel croise (reecrire la table de l'autre, ou une table commune) est hors perimetre -- c'est une strate encore au-dessus.
- Les dettes de verification du chantier (passage 576 vers 144, tore a 37 trous, references arXiv non ouvertes firsthand) restent **RAPPORTEES** et ne sont pas utilisees ici : ce notebook ne derive que du substrat exact de GT-3b, lui-meme exhaustivement verifie.

## 4. Le parcours complet — du jeu nommé au coût de la méta-action

**Versant d'intégration du chantier #12207, absorbé ici en seconde partie.** Les versants précédents ont livré chacun une pièce : [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) construit et vérifie des chemins minimaux entre chambres, [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) identifie les murs où vivent les égalités, la Partie I ci-dessus (marches 1-3) a tarifé les swaps comme des actions que l'agent paie. Cette seconde partie referme la boucle : il réunit les trois pièces **dans un seul parcours**, celui que le chantier demande de bout en bout --

> un lecteur peut, dans un notebook exécuté : partir d'un **jeu nommé**, atteindre un autre jeu nommé par un **chemin qu'il n'a pas écrit lui-même**, **voir le mur** qu'il traverse à chaque pas, et **lire le coût** de la méta-action qui l'y a mené.

## Plan

1. **Le chemin que le lecteur n'écrit pas** : constructeur BFS et vérificateur indépendant (patron GT-20)
2. **Voir les murs** : chaque pas traverse un jeu à égalité codimension 1, dont on exhibe les deux faces
3. **Le coût des méta-actions** : tarifs par côté puis par niveau -- et un résultat inattendu, mesuré
4. **Le parcours complet, bout en bout** : le bloc intégral, puis sa re-vérification indépendante
5. **Exercices**

## La question

La théorie des jeux ordinaire décrit des agents *dans* des règles. La strate méta demande ce qui se passe quand **déplacer le jeu est une action payée** : quel trajet, quels murs franchis, à quel prix, et payé par qui. Sur l'univers fini de Robinson-Gofforth, chaque pièce de cette question est calculable -- ce notebook les calcule **ensemble**, sur des jeux portant un nom (Dilemme, Poule, Cerf), pour que le résultat se lise comme un récit et pas comme une coordonnée.

### 4.0. Le substrat étendu, hérité de GT-3b / GT-20

Un jeu = **deux tables de rangs**, une par joueur, chacune un 4-uplet ordonnant strictement les quatre cases (haut-gauche, haut-droite, bas-gauche, bas-droite) -- rang 4 = meilleur. Les **swaps** échangent les cases portant deux rangs adjacents `k` et `k+1`, de part et d'autre : `R1, R2, R3` (côté Ligne) et `C1, C2, C3` (côté Colonne). Ce sont les six générateurs de l'espace ; chaque traversée d'un swap franchit un **mur** du monde de Bruns-Kimmich -- c'est l'objet de la section 2. Toute la mécanique est reprise telle quelle des trois versants, sans modification : ce notebook est un consommateur du substrat, pas une refonte.

In [11]:
// Substrat GT-3b / GT-3e / GT-20, repris tel quel -- pur BCL
// swap_valeurs_adjacentes : meme corps que SwapVal (cellule 2), nom conserve de la source Python.
Tab SwapValeursAdjacentes(Tab t, int k) => SwapVal(t, k);

Jeu SwapJeu(Jeu jeu, string cote, int k)
{
    // Applique le swap de niveau k sur LA table du joueur designe (meta-action unilaterale).
    if (cote == "ligne") return (SwapValeursAdjacentes(jeu.Item1, k), jeu.Item2);
    return (jeu.Item1, SwapValeursAdjacentes(jeu.Item2, k));
}

// Les 576 jeux stricts : le produit cartesien des 24 tables, cote Ligne x cote Colonne.
var stricts = (from a in Enumerable.Range(1, 4)
               from b in Enumerable.Range(1, 4)
               from c in Enumerable.Range(1, 4)
               from d in Enumerable.Range(1, 4)
               where new[] { a, b, c, d }.Distinct().Count() == 4
               orderby a, b, c, d
               select (a, b, c, d)).ToList();
var chambresJeux = (from r in stricts from c in stricts select (r, c)).ToList();
Console.WriteLine($"Tables strictes par joueur : {stricts.Count} | chambres (jeux stricts) : {chambresJeux.Count}");
Console.WriteLine("Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)");

// Les cinq bornes canoniques (convention GT-21 / GT-3b / GT-20).
// PD est deja declare en cellule 2 (meme valeur) : on le reutilise tel quel.
var POULE = ((3, 2, 4, 1), (3, 4, 2, 1));   // T>R>S>P
var CERF = ((4, 1, 3, 2), (4, 3, 1, 2));    // R>T>P>S
var IDENTITE = ((1, 2, 3, 4), (1, 2, 3, 4));
var RENVERSE = ((4, 3, 2, 1), (4, 3, 2, 1));
var bornes27 = new List<(string Nom, Jeu J)>
{
    ("PD (Dilemme)", PD), ("POULE (Poule)", POULE), ("CERF (Cerf)", CERF),
    ("IDENTITE", IDENTITE), ("RENVERSE", RENVERSE)
};
foreach (var (nom27, j27) in bornes27)
{
    bool ok27 = chambresJeux.Contains(j27);
    Console.WriteLine(Fi($"  {nom27,-16}") + $" Ligne {PyTup(j27.Item1)} | Colonne {PyTup(j27.Item2)} | chambre stricte : {PyBool(ok27)}");
}

Tables strictes par joueur : 24 | chambres (jeux stricts) : 576


Générateurs : R1 R2 R3 (Ligne) x C1 C2 C3 (Colonne)


  PD (Dilemme)     Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2) | chambre stricte : True


  POULE (Poule)    Ligne (3, 2, 4, 1) | Colonne (3, 4, 2, 1) | chambre stricte : True


  CERF (Cerf)      Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2) | chambre stricte : True


  IDENTITE         Ligne (1, 2, 3, 4) | Colonne (1, 2, 3, 4) | chambre stricte : True


  RENVERSE         Ligne (4, 3, 2, 1) | Colonne (4, 3, 2, 1) | chambre stricte : True


### Lecture du substrat

Vingt-quatre tables par joueur, donc 24 x 24 = **576 chambres** -- l'univers de Robinson-Gofforth classique, déjà dérivé par GT-20 et non récité ici. Les cinq bornes canoniques y vivent : le Dilemme du Prisonnier (défection mutuelle stable), la Poule (le jeu du poulet), la Chasse au Cerf (coordination sur le meilleur monde commun), et les deux extrêmes structurants Identité et Renversement.

Deux précisions de vocabulaire pour la suite. Un **pas** relie deux chambres adjacentes : il modifie exactement une table, celle du joueur qui agit, par exactement un swap de niveau `k`. Et une **méta-action** (GT-3e) est ce pas *considéré comme payé* : réécrire une préférence déclarée coûte quelque chose, en échelons de rang -- c'est la section 3 qui fixe le barème.

## 5. Le chemin que le lecteur n'écrit pas

Le premier impératif du parcours : le chemin ne doit pas être écrit à la main. Si le lecteur choisissait lui-même ses swaps, le notebook ne montrerait rien -- il redirait ses propres intuitions. Le constructeur ci-dessous produit donc le chemin par **BFS avec remontée des parents**, exactement selon le patron de GT-20 : le graphe des 576 chambres est exploré depuis le départ, et la chaîne est reconstruite en remontant de l'arrivée.

Le second impératif est la **séparation constructeur / vérificateur** (loi II de la série) : celui qui produit le témoin n'est pas celui qui le juge. La cellule suivante héberge un vérificateur qui ne réutilise *rien* du constructeur -- il re-dérive les distances par sa propre recherche exhaustive.

In [12]:
// === Section 1.1 : construire_chemin -- le constructeur de temoin (patron GT-20) ===

// pred[ti] / cur[ti] : la table du cote ti (0 = Ligne, 1 = Colonne).
Tab TableDe(Jeu jeu, int ti) => ti == 0 ? jeu.Item1 : jeu.Item2;

List<(Jeu Pred, Jeu Cur)> ConstruireChemin(Jeu depart, Jeu arrivee)
{
    // Produit une suite (jeu_avant, jeu_apres) de depart a arrivee par BFS + remontee des parents.
    var parent = new Dictionary<Jeu, Jeu?>();
    parent[depart] = null;
    var q = new Queue<Jeu>();
    q.Enqueue(depart);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        if (u == arrivee) break;
        foreach (string cote in new[] { "ligne", "colonne" })
            foreach (int k in new[] { 1, 2, 3 })
            {
                var v = SwapJeu(u, cote, k);
                if (!parent.ContainsKey(v)) { parent[v] = u; q.Enqueue(v); }
            }
    }
    if (!parent.ContainsKey(arrivee)) return null;
    var chaine = new List<(Jeu, Jeu)>();
    var cur = arrivee;
    while (parent[cur] is not null)
    {
        var prec = parent[cur].Value;
        chaine.Add((prec, cur));
        cur = prec;
    }
    chaine.Reverse();
    return chaine;
}

string NommerPas(Jeu pred, Jeu cur)
{
    // Nomme le pas elementaire entre deux jeux consecutifs (cote, niveaux echanges).
    for (int ti = 0; ti < 2; ti++)
    {
        string cote = ti == 0 ? "ligne" : "colonne";
        if (TableDe(pred, ti) != TableDe(cur, ti))
            foreach (int k in new[] { 1, 2, 3 })
                if (SwapJeu(pred, cote, k) == cur)
                {
                    string nomCote = cote == "ligne" ? "R" : "C";
                    return $"{nomCote}{k} (côté {cote}, niveaux {k} <-> {k + 1})";
                }
    }
    return "?";
}

var cheminPdCerf = ConstruireChemin(PD, CERF);
Console.WriteLine("Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf");
Console.WriteLine($"  départ  PD   : Ligne {PyTup(PD.Item1)} | Colonne {PyTup(PD.Item2)}");
foreach (var (pred, cur) in cheminPdCerf)
    Console.WriteLine($"  -- {NommerPas(pred, cur)} -->  Ligne {PyTup(cur.Item1)} | Colonne {PyTup(cur.Item2)}");
Console.WriteLine($"Longueur du chemin produit : {cheminPdCerf.Count} pas");

Chemin construit : Dilemme du Prisonnier -> Chasse au Cerf


  départ  PD   : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)


  -- R3 (côté ligne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (3, 4, 1, 2)


  -- C3 (côté colonne, niveaux 3 <-> 4) -->  Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)


Longueur du chemin produit : 2 pas


### Lecture : deux révisions du sommet

Le chemin construit a **2 pas** -- et chacun réécrit le **sommet** des préférences d'un joueur (niveaux 3 <-> 4). Ce n'est pas un choix du constructeur, c'est la géométrie : pour passer du Dilemme au Cerf, Ligne doit échanger ses deux meilleures cases (sa table `(3, 1, 4, 2)` devient `(4, 1, 3, 2)`) et Colonne doit faire de même. En langage des noms : dans le Dilemme, la défection est le sommet de chacun ; dans le Cerf, c'est la coordination. Aucun détour par les bas niveaux ne raccourcit ce trajet -- il faut toucher au sommet, deux fois.

On notera l'asymétrie apparente des tables du Cerf (`(4, 1, 3, 2)` et `(4, 3, 1, 2)`) : les deux joueurs y placent le même sommet mais ne classent pas pareil les deux pires cases -- le Cerf n'exige pas d'êtres identiques, seulement de préférer le monde commun.

In [13]:
// === Section 1.2 : verifier_chemin -- le verificateur independant (patron GT-20) ===

Dictionary<Jeu, int> BfsComplet(Jeu depart)
{
    // Distances exhaustives depuis depart (la preuve de minimalite, recalculee par le verificateur).
    var dist = new Dictionary<Jeu, int> { [depart] = 0 };
    var q = new Queue<Jeu>();
    q.Enqueue(depart);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (string cote in new[] { "ligne", "colonne" })
            foreach (int k in new[] { 1, 2, 3 })
            {
                var v = SwapJeu(u, cote, k);
                if (!dist.ContainsKey(v)) { dist[v] = dist[u] + 1; q.Enqueue(v); }
            }
    }
    return dist;
}

bool PasElementaireValide(Jeu pred, Jeu cur)
{
    // True ssi cur s'obtient de pred par exactement un swap elementaire.
    foreach (string cote in new[] { "ligne", "colonne" })
        foreach (int k in new[] { 1, 2, 3 })
            if (SwapJeu(pred, cote, k) == cur) return true;
    return false;
}

string VerifierChemin(Jeu depart, Jeu arrivee, List<(Jeu Pred, Jeu Cur)> chaine)
{
    // Verdict independant : re-derive TOUT, ne reutilise rien du constructeur.
    if (chaine is null || chaine.Count == 0) return "INVALIDE : chemin vide";
    if (chaine[0].Pred != depart || chaine[^1].Cur != arrivee)
        return "INVALIDE : les extrémités ne sont pas celles annoncées";
    foreach (var (pred, cur) in chaine)
        if (!PasElementaireValide(pred, cur))
            return $"INVALIDE : pas non élémentaire vers {PyJeu(cur)}";
    int dReel = BfsComplet(depart)[arrivee];
    if (chaine.Count != dReel)
        return $"VALIDE mais NON MINIMAL : {chaine.Count} pas, distance réelle {dReel}";
    return $"VALIDE + MINIMAL : {chaine.Count} pas = distance exhaustive d(G,H)";
}

Console.WriteLine("1. Le chemin du constructeur (PD -> CERF) :");
Console.WriteLine("   " + VerifierChemin(PD, CERF, cheminPdCerf));

var voisinCerf32 = SwapJeu(CERF, "ligne", 1);
var cheminDetour32 = cheminPdCerf.Concat(new[] { (CERF, voisinCerf32), (voisinCerf32, CERF) }).ToList();
Console.WriteLine("2. Le même chemin rallongé d'un aller-retour :");
Console.WriteLine("   " + VerifierChemin(PD, CERF, cheminDetour32));

Console.WriteLine("3. Un 'chemin' en un pas vers un jeu non adjacent :");
Console.WriteLine("   " + VerifierChemin(PD, POULE, new List<(Jeu Pred, Jeu Cur)> { (PD, POULE) }));

1. Le chemin du constructeur (PD -> CERF) :


   VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)


2. Le même chemin rallongé d'un aller-retour :


   VALIDE mais NON MINIMAL : 4 pas, distance réelle 2


3. Un 'chemin' en un pas vers un jeu non adjacent :


   INVALIDE : pas non élémentaire vers ((3, 2, 4, 1), (3, 4, 2, 1))


### Lecture : trois verdicts, une séparation

Le vérificateur rend trois verdicts distincts sur trois cas construits pour eux : `VALIDE + MINIMAL` pour le témoin du constructeur, `NON MINIMAL` pour le même chemin artificiellement rallongé d'un aller-retour, et `INVALIDE` pour un saut direct PD -> POULE (deux jeux que rien ne relie en un pas). La minimalité n'est pas une impression : elle est **re-dérivée par recherche exhaustive** depuis le départ, indépendamment de la façon dont le constructeur a trouvé son chemin.

C'est la loi II de la série, appliquée au parcours : le constructeur (BFS + parents) *produit* un témoin, le vérificateur (re-dérivation complète) le *juge*. À la section 4, ce couple sera étendu aux murs et aux coûts : le vérificateur final re-vérifiera tout ce que le parcours affiche.

## 6. Voir les murs

Entre deux chambres adjacentes -- avant et après un swap de niveau `k` -- il y a un **mur** : un jeu à rangs où les deux niveaux `k` et `k+1` sont tombés **ex æquo**. C'est la lecture Bruns-Kimmich reprise de GT-3b : le monde complet des jeux à rangs compte 5625 points, dont 576 chambres (aucune égalité) et des murs de codimension 1 (exactement une paire ex æquo). Traverser un swap, c'est littéralement *passer à travers* le mur qui sépare les deux chambres.

Deux opérations réciproques le formalisent : **fusionner** les niveaux `k` et `k+1` d'une chambre donne le mur qu'un pas de niveau `k` traverse ; **briser le tie** d'un mur redonne les deux chambres qu'il sépare. La propriété qui fait tenir la section : briser le tie du mur d'un pas doit redonner *exactement* les deux tables extrémités de ce pas. Elle est vérifiée globalement ci-dessous, sur les 24 tables et les 3 niveaux.

In [14]:
// === Section 2.1 : le mur d'un pas -- fusion de niveaux et brisure de tie (GT-3b) ===

Tab MurDuNiveau(Tab t, int k)
{
    // L'ordre faible (codim 1) ou les niveaux k et k+1 de t sont ex aequo : le mur du pas de niveau k.
    var orig = new[] { At(t, 0), At(t, 1), At(t, 2), At(t, 3) };
    var l = orig.Select(x => x > k ? x - 1 : x).ToArray();   // les niveaux au-dessus descendent d'un cran
    l[Array.IndexOf(orig, k)] = k;
    l[Array.IndexOf(orig, k + 1)] = k;                       // les deux niveaux fusionnent en k
    return (l[0], l[1], l[2], l[3]);
}

(Tab A, Tab B) BriserLeTie(Tab t)
{
    // Les deux chambres strictes que le mur t separe (reciproque exacte de MurDuNiveau).
    var vals = new[] { At(t, 0), At(t, 1), At(t, 2), At(t, 3) };
    int v = vals.Where(x => vals.Count(y => y == x) > 1).Min();   // la valeur ex aequo (mur simple : unique)
    var pos = Enumerable.Range(0, 4).Where(p => vals[p] == v).ToArray();
    int i = pos[0], j = pos[1];
    var inc = vals.Select(x => x > v ? x + 1 : x).ToArray();      // les niveaux au-dessus remontent d'un cran
    var A = (int[])inc.Clone();
    var B = (int[])inc.Clone();
    A[i] = v; A[j] = v + 1;
    B[i] = v + 1; B[j] = v;
    return ((A[0], A[1], A[2], A[3]), (B[0], B[1], B[2], B[3]));
}

// Propriete cle, verifiee GLOBALEMENT : briser le tie du mur d'un pas redonne ses extremites.
bool prop35 = stricts.All(t => new[] { 1, 2, 3 }.All(k =>
{
    var (fa, fb) = BriserLeTie(MurDuNiveau(t, k));
    var cote = new[] { fa, fb }.OrderBy(x => x).ToArray();
    var extremites = new[] { t, SwapValeursAdjacentes(t, k) }.OrderBy(x => x).ToArray();
    return cote.SequenceEqual(extremites);
}));
Console.WriteLine($"Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : {PyBool(prop35)}");

var mursParJoueur35 = new HashSet<Tab>(stricts.SelectMany(t => new[] { 1, 2, 3 }.Select(k => MurDuNiveau(t, k))));
Console.WriteLine($"Murs simples distincts par joueur : {mursParJoueur35.Count} (GT-3b mesurait 36)");

var t35 = PD.Item1;
var mur35 = MurDuNiveau(t35, 3);
Console.WriteLine($"Exemple : table Ligne du Dilemme {PyTup(t35)} | pas de niveau 3");
Console.WriteLine($"  mur          : {PyTup(mur35)} (codim {4 - DistinctCount(mur35)})");
var (faceA35, faceB35) = BriserLeTie(mur35);
Console.WriteLine($"  faces du mur : {PyTup(faceA35)}, {PyTup(faceB35)}");
Console.WriteLine($"  extrémités   : {PyTup(t35)} et {PyTup(SwapValeursAdjacentes(t35, 3))}");

Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : True


Murs simples distincts par joueur : 36 (GT-3b mesurait 36)


Exemple : table Ligne du Dilemme (3, 1, 4, 2) | pas de niveau 3


  mur          : (3, 1, 3, 2) (codim 1)


  faces du mur : (3, 1, 4, 2), (4, 1, 3, 2)


  extrémités   : (3, 1, 4, 2) et (4, 1, 3, 2)


### Lecture : le mur a deux faces, et ce sont les extrémités du pas

La propriété est vérifiée sur les 72 combinaisons (24 tables x 3 niveaux) : **briser le tie du mur redonne exactement la chambre de départ et la chambre d'arrivée du pas**. Le mur n'est donc pas une métaphore décorative -- c'est l'objet géométrique *entre* les deux jeux, celui dont les deux faces sont les extrémités du pas. Sur l'exemple affiché : la table Ligne du Dilemme `(3, 1, 4, 2)` et sa voisine de niveau 3 `(4, 1, 3, 2)` -- la table du Cerf -- sont les deux faces du mur `(3, 1, 3, 2)`, où les cases haut-gauche et bas-gauche sont ex æquo au sommet.

Le comptage rejoint GT-3b : **36 murs simples par joueur** (24 chambres x 3 niveaux, chaque mur compté deux fois car chaque mur touche exactement 2 chambres). Chaque niveau de préférence a ses murs ; franchir un mur de niveau 3, c'est réécrire un sommet -- le lien avec le coût de la section suivante est direct.

In [15]:
// === Section 2.2 : les murs du chemin PD -> CERF, pas a pas ===

Console.WriteLine("Le chemin PD -> CERF, lu mur par mur :");
for (int n37 = 0; n37 < cheminPdCerf.Count; n37++)
{
    var (pred37, cur37) = cheminPdCerf[n37];
    int ti37 = pred37.Item1 != cur37.Item1 ? 0 : 1;
    string cote37 = ti37 == 0 ? "ligne" : "colonne";
    int k37 = new[] { 1, 2, 3 }.First(kk => SwapJeu(pred37, cote37, kk) == cur37);
    var mur37 = MurDuNiveau(TableDe(pred37, ti37), k37);
    var (fa37, fb37) = BriserLeTie(mur37);
    bool confirmee37 = new[] { fa37, fb37 }.OrderBy(x => x).SequenceEqual(new[] { TableDe(pred37, ti37), TableDe(cur37, ti37) }.OrderBy(x => x));
    Console.WriteLine($"  pas {n37 + 1} : {NommerPas(pred37, cur37)}");
    Console.WriteLine($"     table {cote37} : {PyTup(TableDe(pred37, ti37))} -> {PyTup(TableDe(cur37, ti37))}");
    Console.WriteLine($"     mur traversé : {PyTup(mur37)} (codim {4 - DistinctCount(mur37)}) | faces : {PyTup(fa37)}, {PyTup(fb37)}");
    Console.WriteLine($"     les faces sont les extrémités du pas : {PyBool(confirmee37)}");
}

Le chemin PD -> CERF, lu mur par mur :


  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)


     table ligne : (3, 1, 4, 2) -> (4, 1, 3, 2)


     mur traversé : (3, 1, 3, 2) (codim 1) | faces : (3, 1, 4, 2), (4, 1, 3, 2)


     les faces sont les extrémités du pas : True


  pas 2 : C3 (côté colonne, niveaux 3 <-> 4)


     table colonne : (3, 4, 1, 2) -> (4, 3, 1, 2)


     mur traversé : (3, 3, 1, 2) (codim 1) | faces : (3, 4, 1, 2), (4, 3, 1, 2)


     les faces sont les extrémités du pas : True


### Lecture : deux murs, un par joueur

Le chemin PD -> CERF traverse exactement **deux murs**, un par joueur, tous deux de niveau 3. Le mur de Ligne, `(3, 1, 3, 2)`, est le jeu où Ligne est *indifférent entre défection et coopération* (ses cases haut-gauche et bas-gauche ex æquo au sommet) tandis que la table de Colonne est encore celle du Dilemme. Le mur de Colonne, `(3, 3, 1, 2)`, est le miroir : Colonne indifférent au sommet, Ligne déjà converti au Cerf.

C'est le contenu concret du critère d'intégration : le chemin n'est plus une suite de coordonnées, c'est une suite de **situations intermédiaires lisibles** -- chacune est un jeu où un joueur exactement est suspendu entre ses deux anciens idéaux. La ligne `confirmee : True` de chaque pas est la garantie que le mur exhibé est bien *celui-là* et pas un voisin.

## 7. Le coût des méta-actions

Chaque pas du chemin est une **méta-action** au sens de GT-3e : un joueur réécrit ses préférences déclarées, et cette réécriture est payée en échelons de rang -- l'unité honnête d'un monde purement ordinal (payer 1, c'est renoncer à un échelon pour atteindre le niveau visé). Reste à fixer le **barème**. Deux lectures naturelles :

- **par côté** : Ligne paie `c_L` par swap, Colonne `c_C` -- le tarif différencié, par exemple la réécriture de la table Colonne (l'"institution") plus cher que celle de Ligne ;
- **par niveau** : réécrire le bas de sa liste coûte 1, le milieu 2, le **sommet 3** -- réviser ce qu'on préfère *entre tout* est la révision la plus profonde.

La question qui semble s'imposer : le chemin le plus court en nombre de pas est-il le moins cher ? L'intuition crie *non* -- un détour par des niveaux bon marché pourrait battre le trajet direct. La mesure, elle, répond autrement.

In [16]:
// === Section 3.1 : tarifs par cote -- le cout est geometrique ===

Dictionary<Tab, int> DistTables(Tab t0)
{
    // Distances du graphe des 24 tables d'UN joueur (generateurs : les 3 swaps de niveau).
    var dist = new Dictionary<Tab, int> { [t0] = 0 };
    var q = new Queue<Tab>();
    q.Enqueue(t0);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (int k in new[] { 1, 2, 3 })
        {
            var v = SwapValeursAdjacentes(u, k);
            if (!dist.ContainsKey(v)) { dist[v] = dist[u] + 1; q.Enqueue(v); }
        }
    }
    return dist;
}

var NOMS40 = new List<(string Nom, Jeu J)>
{
    ("PD", PD), ("POULE", POULE), ("CERF", CERF), ("IDENTITE", IDENTITE), ("RENVERSE", RENVERSE)
};
int cL40 = 1, cC40 = 3;
Console.WriteLine($"Tarifs par côté : Ligne {cL40}, Colonne {cC40}");
Console.WriteLine("paire".PadLeft(22) + " | d_L | d_C | longueur | coût | Ligne paie | Colonne paie");
Console.WriteLine(new string('-', 78));
for (int i40 = 0; i40 < NOMS40.Count; i40++)
    for (int j40 = i40 + 1; j40 < NOMS40.Count; j40++)
    {
        var (n1_40, j1_40) = NOMS40[i40];
        var (n2_40, j2_40) = NOMS40[j40];
        int dL40 = DistTables(j1_40.Item1)[j2_40.Item1];
        int dC40 = DistTables(j1_40.Item2)[j2_40.Item2];
        Console.WriteLine($"{n1_40,10} -> {n2_40,-10} | {dL40}  | {dC40}  |    {dL40 + dC40}     |  {cL40 * dL40 + cC40 * dC40,2}  |     {cL40 * dL40}      |      {cC40 * dC40}");
    }

Tarifs par côté : Ligne 1, Colonne 3


                 paire | d_L | d_C | longueur | coût | Ligne paie | Colonne paie


------------------------------------------------------------------------------


        PD -> POULE      | 1  | 1  |    2     |   4  |     1      |      3


        PD -> CERF       | 1  | 1  |    2     |   4  |     1      |      3


        PD -> IDENTITE   | 3  | 4  |    7     |  15  |     3      |      12


... (3 outputs left out)

     POULE -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3


      CERF -> IDENTITE   | 4  | 5  |    9     |  19  |     4      |      15


      CERF -> RENVERSE   | 2  | 1  |    3     |   5  |     2      |      3


  IDENTITE -> RENVERSE   | 6  | 6  |    12     |  24  |     6      |      18


### Lecture : un théorème, pas une coïncidence

Les colonnes `d_L` et `d_C` sont les distances **dans le graphe des 24 tables d'un seul joueur** -- chacune recalculée par BFS indépendant. Et le tableau montre la structure : **la longueur du chemin minimal est exactement `d_L + d_C`**, et son coût exactement `c_L·d_L + c_C·d_C`. Ce n'est pas un accident d'échantillon, c'est un théorème de trois lignes :

> chaque pas modifie exactement **une** table ; la table de Ligne doit passer de `PD[0]` à `CERF[0]`, ce qui exige au moins `d_L` pas côté Ligne ; symétriquement au moins `d_C` côté Colonne. Tout chemin a donc au moins `d_L + d_C` pas et coûte au moins `c_L·d_L + c_C·d_C` -- et le chemin BFS atteint exactement ces deux bornes.

La conséquence est le résultat central de cette section : **le coût du trajet minimal n'est pas négociable**. Aucun choix de chemin -- aucune ruse, aucun détour -- ne change la facture : qui paie quoi est fixé par la géométrie des deux tables, avant tout parcours. Pour aller du Dilemme au Cerf à ces tarifs, Ligne paie `c_L` et Colonne `c_C` -- quel que soit le chemin minimal emprunté.

In [17]:
// === Section 3.2 : tarifs par niveau -- la mesure repond a l'intuition ===

var W = new Dictionary<int, int> { [1] = 1, [2] = 2, [3] = 3 };   // bas 1, milieu 2, sommet 3

int CoutChemin(List<(Jeu Pred, Jeu Cur)> chaine, Dictionary<int, int> w = null)
{
    w ??= W;
    int total = 0;
    foreach (var (pred, cur) in chaine)
    {
        int ti = pred.Item1 != cur.Item1 ? 0 : 1;
        string cote = ti == 0 ? "ligne" : "colonne";
        int k = new[] { 1, 2, 3 }.First(kk => SwapJeu(pred, cote, kk) == cur);
        total += w[k];
    }
    return total;
}

List<(Jeu Pred, Jeu Cur)> CheminMinCout(Jeu depart, Jeu arrivee, Dictionary<int, int> w = null)
{
    // Dijkstra : le chemin de cout minimal sous les tarifs w (pas forcement le plus court).
    w ??= W;
    var dist = new Dictionary<Jeu, int> { [depart] = 0 };
    var parent = new Dictionary<Jeu, Jeu?> { [depart] = null };
    var pq = new PriorityQueue<Jeu, (int, Jeu)>();
    pq.Enqueue(depart, (0, depart));
    while (pq.Count > 0)
    {
        pq.TryDequeue(out var u, out var pr);
        int d = pr.Item1;
        if (u == arrivee) break;
        if (d > dist.GetValueOrDefault(u, int.MaxValue)) continue;
        foreach (string cote in new[] { "ligne", "colonne" })
            foreach (int k in new[] { 1, 2, 3 })
            {
                var v = SwapJeu(u, cote, k);
                int nd = d + w[k];
                if (nd < dist.GetValueOrDefault(v, int.MaxValue))
                {
                    dist[v] = nd;
                    parent[v] = u;
                    pq.Enqueue(v, (nd, v));   // priorite (distance, jeu) : meme ordre de pop que heapq
                }
            }
    }
    var chaine = new List<(Jeu, Jeu)>();
    var cur = arrivee;
    while (parent[cur] is not null)
    {
        var prec = parent[cur].Value;
        chaine.Add((prec, cur));
        cur = prec;
    }
    chaine.Reverse();
    return chaine;
}

Console.WriteLine($"Tarifs par niveau : {PyDictInt(W)} | le plus court chemin est-il le moins cher ?");
int ecarts42 = 0;
for (int i42 = 0; i42 < NOMS40.Count; i42++)
    for (int j42 = i42 + 1; j42 < NOMS40.Count; j42++)
    {
        var (n1_42, j1_42) = NOMS40[i42];
        var (n2_42, j2_42) = NOMS40[j42];
        var bfs42 = ConstruireChemin(j1_42, j2_42);
        var dij42 = CheminMinCout(j1_42, j2_42);
        int cb42 = CoutChemin(bfs42), cd42 = CoutChemin(dij42);
        int ecart42 = cb42 - cd42;
        if (ecart42 > 0) ecarts42 += 1;
        Console.WriteLine($"  {n1_42,8} -> {n2_42,-8} : BFS {bfs42.Count} pas à {cb42} | Dijkstra {dij42.Count} pas à {cd42} | écart {ecart42}");
    }
Console.WriteLine($"Paires nommées où un détour battrait le plus court chemin : {ecarts42} / 10");

// random.seed(0) / random.choice : port BCL du MT19937 de CPython (init_by_array, tempering).
// System.Random n'est PAS MT19937 : l'echantillon de 300 paires ne serait pas celui du carnet Python.
var mt = new uint[624];
int mtIndex = 624;

void InitGenrand(uint s)
{
    mt[0] = s;
    for (int i = 1; i < 624; i++)
        mt[i] = unchecked(1812433253U * (mt[i - 1] ^ (mt[i - 1] >> 30)) + (uint)i);
    mtIndex = 624;
}

void InitByArray(uint[] key)
{
    InitGenrand(19650218U);
    int i = 1, j = 0;
    int k = 624 > key.Length ? 624 : key.Length;
    for (; k > 0; k--)
    {
        mt[i] = unchecked((mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)) * 1664525U)) + key[j] + (uint)j);
        i++; j++;
        if (i >= 624) { mt[0] = mt[623]; i = 1; }
        if (j >= key.Length) j = 0;
    }
    for (k = 623; k > 0; k--)
    {
        mt[i] = unchecked((mt[i] ^ ((mt[i - 1] ^ (mt[i - 1] >> 30)) * 1566083941U)) - (uint)i);
        i++;
        if (i >= 624) { mt[0] = mt[623]; i = 1; }
    }
    mt[0] = 0x80000000U;
}

uint GenrandUint32()
{
    if (mtIndex >= 624)
    {
        for (int kk = 0; kk < 227; kk++)
        {
            uint y = (mt[kk] & 0x80000000U) | (mt[kk + 1] & 0x7fffffffU);
            mt[kk] = mt[kk + 397] ^ (y >> 1) ^ ((y & 1U) != 0 ? 0x9908b0dfU : 0U);
        }
        for (int kk = 227; kk < 623; kk++)
        {
            uint y = (mt[kk] & 0x80000000U) | (mt[kk + 1] & 0x7fffffffU);
            mt[kk] = mt[kk - 227] ^ (y >> 1) ^ ((y & 1U) != 0 ? 0x9908b0dfU : 0U);
        }
        uint yl = (mt[623] & 0x80000000U) | (mt[0] & 0x7fffffffU);
        mt[623] = mt[396] ^ (yl >> 1) ^ ((yl & 1U) != 0 ? 0x9908b0dfU : 0U);
        mtIndex = 0;
    }
    uint y2 = mt[mtIndex++];
    y2 ^= y2 >> 11;
    y2 ^= (y2 << 7) & 0x9d2c5680U;
    y2 ^= (y2 << 15) & 0xefc60000U;
    y2 ^= y2 >> 18;
    return y2;
}

int RandBelow(int n)
{
    // _randbelow_with_getrandbits de CPython : k = n.bit_length(), rejet tant que r >= n.
    int nbits = 32 - System.Numerics.BitOperations.LeadingZeroCount((uint)n);
    int r = (int)(GenrandUint32() >> (32 - nbits));
    while (r >= n) r = (int)(GenrandUint32() >> (32 - nbits));
    return r;
}

T PyChoice<T>(IList<T> seq) => seq[RandBelow(seq.Count)];

InitByArray(new uint[] { 0 });   // random.seed(0)
var ech42 = new List<(Jeu A, Jeu B)>();
for (int tirage42 = 0; tirage42 < 300; tirage42++)
{
    var a42 = PyChoice(chambresJeux);
    var b42 = PyChoice(chambresJeux);
    ech42.Add((a42, b42));
}
int disc42 = ech42.Count(p => CoutChemin(CheminMinCout(p.A, p.B)) < CoutChemin(ConstruireChemin(p.A, p.B)));
Console.WriteLine($"Échantillon de 300 paires quelconques de chambres : {disc42} détour(s) gagnant(s)");

// La non-unicite du temoin : deux chemins distincts, la meme facture
var bfsPi42 = ConstruireChemin(PD, IDENTITE);
var dijPi42 = CheminMinCout(PD, IDENTITE);
Console.WriteLine();
Console.WriteLine("PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts");
var bfsNoms42 = PyLst(bfsPi42.Take(4), p => PyStr(NommerPas(p.Pred, p.Cur)));
var dijNoms42 = PyLst(dijPi42.Take(4), p => PyStr(NommerPas(p.Pred, p.Cur)));
Console.WriteLine($"  BFS       : {bfsNoms42} ... coût {CoutChemin(bfsPi42)}");
Console.WriteLine($"  Dijkstra  : {dijNoms42} ... coût {CoutChemin(dijPi42)}");

Tarifs par niveau : {1: 1, 2: 2, 3: 3} | le plus court chemin est-il le moins cher ?


        PD -> POULE    : BFS 2 pas à 2 | Dijkstra 2 pas à 2 | écart 0


        PD -> CERF     : BFS 2 pas à 6 | Dijkstra 2 pas à 6 | écart 0


        PD -> IDENTITE : BFS 7 pas à 14 | Dijkstra 7 pas à 14 | écart 0


        PD -> RENVERSE : BFS 5 pas à 10 | Dijkstra 5 pas à 10 | écart 0


     POULE -> CERF     : BFS 4 pas à 8 | Dijkstra 4 pas à 8 | écart 0


... (6 outputs left out)

Échantillon de 300 paires quelconques de chambres : 0 détour(s) gagnant(s)


PD -> IDENTITE : le BFS et le Dijkstra produisent deux chemins distincts


  BFS       : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'R3 (côté ligne, niveaux 3 <-> 4)', 'C2 (côté colonne, niveaux 2 <-> 3)'] ... coût 14


  Dijkstra  : ['R2 (côté ligne, niveaux 2 <-> 3)', 'R1 (côté ligne, niveaux 1 <-> 2)', 'C2 (côté colonne, niveaux 2 <-> 3)', 'C1 (côté colonne, niveaux 1 <-> 2)'] ... coût 14


### Lecture : le plus court est aussi le moins cher -- mesuré, pas prouvé

L'intuition du détour économique est **réfutée par la mesure** : sur les 10 paires nommées et sur un échantillon de 300 paires quelconques de chambres, **aucun** chemin de coût minimal n'est plus long que le plus court chemin. Sous ces tarifs, la longueur minimale et le coût minimal sont atteints *simultanément* -- le Dijkstra et le BFS rendent la même facture. Ce fait est **mesuré**, pas démontré : la section 3.1 le prouve pour les tarifs par côté (théorème des bornes `d_L`, `d_C`), mais pour les tarifs par niveau, aucun argument court ne clôt la question -- c'est dit tel quel, et l'exercice 2 la rouvre.

Le dernier affichage montre l'autre face de la structure : pour PD -> IDENTITE, BFS et Dijkstra produisent **deux chemins distincts de même coût**. Le témoin n'est pas unique -- plusieurs routes mènent au même prix -- mais la facture, elle, ne varie pas. Réuni au théorème de la section 3.1, le paysage économique du parcours tient en une phrase : **le prix du voyage est un invariant géométrique ; le chemin, lui, est un choix parmi des égalités**.

## 8. Le parcours complet, bout en bout

Les trois pièces sont prêtes : le constructeur de chemin (section 1), les murs (section 2), le barème (section 3). La fonction ci-dessous les assemble dans **un seul bloc** -- celui que le critère d'intégration du chantier demande : un jeu nommé, un chemin construit et non écrit à la main, chaque mur traversé exhibé avec ses deux faces, chaque méta-action affichée avec son coût et son payeur, et la facture finale par joueur.

Le barème est celui de la section 3.2 (bas 1, milieu 2, sommet 3), le même pour les deux joueurs -- la profondeur de la révision fait le prix, pas l'identité de qui révise.

In [18]:
// === Section 4.1 : parcours_complet -- le bloc du critere d'integration ===

List<(Jeu Pred, Jeu Cur)> ParcoursComplet(Jeu depart, Jeu arrivee, Dictionary<int, int> w, string nomDep, string nomArr)
{
    // Le parcours integral : chemin construit + murs traverses + couts, affiches d'un seul bloc.
    var chaine = ConstruireChemin(depart, arrivee);
    Console.WriteLine($"PARCOURS COMPLET : {nomDep} -> {nomArr}");
    Console.WriteLine($"  barème par niveau : {PyDictInt(w)} (le sommet coûte le plus cher)");
    Console.WriteLine($"  {nomDep} : Ligne {PyTup(depart.Item1)} | Colonne {PyTup(depart.Item2)}");
    int cumul = 0;
    var parJoueur = new Dictionary<string, int> { ["ligne"] = 0, ["colonne"] = 0 };
    for (int n45 = 0; n45 < chaine.Count; n45++)
    {
        var (pred, cur) = chaine[n45];
        int ti = pred.Item1 != cur.Item1 ? 0 : 1;
        string cote = ti == 0 ? "ligne" : "colonne";
        int k = new[] { 1, 2, 3 }.First(kk => SwapJeu(pred, cote, kk) == cur);
        var mur = MurDuNiveau(TableDe(pred, ti), k);
        int cout = w[k];
        cumul += cout;
        parJoueur[cote] += cout;
        var (fa, fb) = BriserLeTie(mur);
        Console.WriteLine($"  pas {n45 + 1} : {NommerPas(pred, cur)}");
        Console.WriteLine($"          jeu {PyTup(pred.Item1)}|{PyTup(pred.Item2)} -> {PyTup(cur.Item1)}|{PyTup(cur.Item2)}");
        Console.WriteLine($"          mur traversé : table {cote} {PyTup(mur)} (codim 1), faces {PyTup(fa)}, {PyTup(fb)}");
        Console.WriteLine($"          coût {cout} payé par {cote} | cumul {cumul}");
    }
    Console.WriteLine($"  {nomArr} : Ligne {PyTup(arrivee.Item1)} | Colonne {PyTup(arrivee.Item2)}");
    int paieLigne45 = parJoueur["ligne"], paieCol45 = parJoueur["colonne"];
    Console.WriteLine($"  ARRIVÉE : {chaine.Count} pas | facture totale {cumul} (Ligne {paieLigne45}, Colonne {paieCol45})");
    return chaine;
}

var cheminFinal45 = ParcoursComplet(PD, CERF, W, "Dilemme (PD)", "Chasse au Cerf (CERF)");

PARCOURS COMPLET : Dilemme (PD) -> Chasse au Cerf (CERF)


  barème par niveau : {1: 1, 2: 2, 3: 3} (le sommet coûte le plus cher)


  Dilemme (PD) : Ligne (3, 1, 4, 2) | Colonne (3, 4, 1, 2)


  pas 1 : R3 (côté ligne, niveaux 3 <-> 4)


          jeu (3, 1, 4, 2)|(3, 4, 1, 2) -> (4, 1, 3, 2)|(3, 4, 1, 2)


          mur traversé : table ligne (3, 1, 3, 2) (codim 1), faces (3, 1, 4, 2), (4, 1, 3, 2)


... (3 outputs left out)

          mur traversé : table colonne (3, 3, 1, 2) (codim 1), faces (3, 4, 1, 2), (4, 3, 1, 2)


          coût 3 payé par colonne | cumul 6


  Chasse au Cerf (CERF) : Ligne (4, 1, 3, 2) | Colonne (4, 3, 1, 2)


  ARRIVÉE : 2 pas | facture totale 6 (Ligne 3, Colonne 3)


### Lecture : le récit complet, lisible d'un bloc

Le bloc répond terme à terme au critère. Le jeu de départ porte un nom (Dilemme) et l'arrivée aussi (Chasse au Cerf). Le chemin est **construit** par BFS -- le lecteur ne l'a pas écrit. Chaque pas affiche **son mur** : la table à égalité exactement entre les deux jeux, avec ses deux faces, qui sont les extrémités du pas. Et chaque pas affiche **son coût** : deux révisions de sommet à 3 échelons chacune, payées séparément -- Ligne paie 3, Colonne paie 3, facture totale 6.

Le récit économique se lit maintenant mot à mot : *sortir du Dilemme vers le Cerf ne coûte aucun bas niveau -- il ne coûte que des sommets*. Personne n'a besoin de réviser ses pires cases ; chacun doit seulement admettre que sa meilleure case change de nature. C'est précisément le contenu de GT-3e sur ce couple (l'indifference exacte de la fuite solitaire), désormais visible comme géométrie tarifée : les deux murs franchis sont tous deux des murs de sommet.

In [19]:
// === Section 4.2 : verifier_parcours -- la re-verification independante, etendue ===

string VerifierParcours(Jeu depart, Jeu arrivee, List<(Jeu Pred, Jeu Cur)> chaine, Dictionary<int, int> w, string nomDep, string nomArr)
{
    // Re-derive TOUT ce que ParcoursComplet affiche : extremites, pas, murs, couts, minimalites.
    if (chaine is null || chaine.Count == 0 || chaine[0].Pred != depart || chaine[^1].Cur != arrivee)
        return "INVALIDE : les extrémités ne sont pas celles annoncées";
    int cumul = 0;
    foreach (var (pred, cur) in chaine)
    {
        int ti = pred.Item1 != cur.Item1 ? 0 : 1;
        string cote = ti == 0 ? "ligne" : "colonne";
        var ks = new[] { 1, 2, 3 }.Where(kk => SwapJeu(pred, cote, kk) == cur).ToList();
        if (ks.Count == 0) return $"INVALIDE : pas non élémentaire vers {PyJeu(cur)}";
        int k = ks[0];
        var (fa, fb) = BriserLeTie(MurDuNiveau(TableDe(pred, ti), k));
        var faces = new[] { fa, fb }.OrderBy(x => x).ToArray();
        var extremites = new[] { TableDe(pred, ti), TableDe(cur, ti) }.OrderBy(x => x).ToArray();
        if (!faces.SequenceEqual(extremites))
            return $"INVALIDE : le mur exhibé ne sépare pas {PyJeu(pred)} et {PyJeu(cur)}";
        cumul += w[k];
    }
    int dLong = BfsComplet(depart)[arrivee];
    if (chaine.Count != dLong)
        return $"INVALIDE : {chaine.Count} pas, distance réelle {dLong}";
    var opt = CheminMinCout(depart, arrivee, w);
    int dCout = CoutChemin(opt, w);
    if (cumul != dCout)
        return $"VALIDE mais NON MINIMAL EN COÛT : {cumul} payés, optimum {dCout}";
    return $"TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : {nomDep} -> {nomArr} | {chaine.Count} pas = distance exhaustive | facture {cumul} = optimum Dijkstra | chaque mur confirmé par ses faces";
}

Console.WriteLine(VerifierParcours(PD, CERF, cheminFinal45, W, "Dilemme (PD)", "Chasse au Cerf (CERF)"));

var voisinCerf47 = SwapJeu(CERF, "ligne", 1);
Console.WriteLine();
Console.WriteLine("Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :");
var rallonge47 = cheminFinal45.Concat(new[] { (CERF, voisinCerf47), (voisinCerf47, CERF) }).ToList();
Console.WriteLine("   " + VerifierParcours(PD, CERF, rallonge47, W, "PD", "CERF"));

TEMOIN CONSTRUIT ET VÉRIFIÉ INDÉPENDAMMENT : Dilemme (PD) -> Chasse au Cerf (CERF) | 2 pas = distance exhaustive | facture 6 = optimum Dijkstra | chaque mur confirmé par ses faces


Contre-épreuve -- le même parcours rallongé d'un aller-retour gratuit pour les yeux :


   INVALIDE : 4 pas, distance réelle 2


### Lecture : le témoin construit et vérifié indépendamment

Le vérificateur étendu ne se contente pas des extrémités et de l'élémentarité : il re-dérive **chaque mur** (brisure de tie recalculée, faces comparées aux extrémités affichées), **chaque coût** (barème réappliqué pas à pas), la **minimalité en longueur** (BFS exhaustif) et la **minimalité en coût** (Dijkstra indépendant) -- et rend son verdict en une ligne. La contre-épreuve finale montre qu'un chemin rallongé d'un aller-retour, invisible aux yeux s'il était bien imprimé, est attrapé par la distance réelle.

La terminologie est celle du steer #12205 : un chemin produit par un constructeur et accepté par un vérificateur séparé est un **témoin construit et vérifié indépendamment** -- pas une preuve au sens formel, mais un objet dont chaque affirmation affichée a été re-dérivée par un code qui ne partage rien avec celui qui l'a produite. C'est le standard de la série depuis GT-20, étendu ici aux murs et aux coûts.

## Exercice 1 -- le sweep vu par Colonne

La marche 2 a fait migrer Ligne. Refaites le meme sweep du cout **cote Colonne** (ses swaps C1-C3, son gain au sorti de la dynamique). Le compte de migrants a c=1 est-il le meme que cote Ligne (16 %) ? Si oui, dites pourquoi la symetrie l'impose ; sinon, exhibez un jeu qui les distingue.

In [20]:
// Exercice 1 : sweep du cout cote Colonne
// TODO etudiant : reconstruire bestByDistCol (BFS sur la table Colonne, payoff cote 1)
// puis le tableau c=0..3 : %migrer, distance optimale moyenne, fuient-un-cycle
Dictionary<int, (int PctMigrer, double DistMoyenne, int Fuites)> resultatEx1 = null;  // TODO etudiant
Console.WriteLine("Exercice 1 : a completer (sweep cote Colonne)");

Exercice 1 : a completer (sweep cote Colonne)


## Exercice 2 -- deplacer la selection d'equilibre

Les 72 jeux a deux equilibres purs sont les coordinations. En partant de l'un d'eux, montrez un cas ou **un seul swap de Ligne change l'equilibre selectionne** par la dynamique (la case d'arrivee change), et un cas ou aucun swap ne la change. Combien des 72 jeux sont du premier type ? (Le comptage exhaustif est possible : 72 jeux x 24 tables.)

In [21]:
// Exercice 2 : la selection d'equilibre sous meta-actions
// TODO etudiant : pour chaque jeu a 2 NE purs, comparer l'arrivee de BrDyn pour la table
// courante et pour les 23 autres tables ; compter ceux ou l'arrivee peut changer
(int Nb, Jeu Exemple)? resultatEx2 = null;  // TODO etudiant
Console.WriteLine("Exercice 2 : a completer (selection deplacable)");

Exercice 2 : a completer (selection deplacable)


## Exercice 3 -- le meta-jeu sequentiel

La marche 3 est simultanee : les 4 echecs de coordination dur proviennent de l'instabilite de l'accord. Rendez le meta-jeu **sequentiel** : Ligne annonce et joue sa meta-action, Colonne voit le jeu deplace puis choisit la sienne. Proposez le protocole (qui observe quoi, ordre des couts), recalculez les equilibres (backward induction sur les 4x4 sous-jeux), et dites combien des 4 echecs durs se referment. Verdict attendu en une phrase : *le sequentiel suffit-il a stabiliser l'accord ?*

In [22]:
// Exercice 3 : equilibres parfaits du meta-jeu sequentiel
// TODO etudiant : pour chacun des 4 jeux en echec dur, construire l'arbre
// (Ligne : 4 actions ; Colonne observe ; 4 actions) et resoudre par induction
Dictionary<Jeu, ((int, int) Equilibre, bool Referme)> resultatEx3 = null;  // TODO etudiant
Console.WriteLine("Exercice 3 : a completer (meta-jeu sequentiel)");

Exercice 3 : a completer (meta-jeu sequentiel)


## Exercice 4 — Le mur du pas, dans l'autre sens

La section 2 allait du pas vers le mur ; cet exercice remonte le chemin inverse. On donne deux chambres adjacentes quelconques `(pred, cur)` et on demande de retrouver le mur traversé : identifier le côté modifié et le niveau `k` (le pas), construire le mur par fusion de niveaux, puis vérifier que `briser_le_tie(mur)` redonne exactement `{pred, cur}` — la propriété de la section 2.

Test attendu sur `pred_ex -> cur_ex` : côté colonne, niveau 1, mur `(2, 3, 1, 1)`.

In [23]:
// Exercice 1 -- le mur du pas, dans l'autre sens
// On donne deux chambres adjacentes quelconques (pred, cur). Retrouver le mur traverse :
// 1. identifier le cote modifie et le niveau k (le pas) ;
// 2. construire le mur par fusion de niveaux ;
// 3. verifier que BriserLeTie(mur) redonne exactement {pred, cur} -- la propriete de la section 2.
// Test attendu sur predEx56 -> curEx56 : cote colonne, niveau 1, mur (2, 3, 1, 1).

(string Cote, int K, Tab Mur)? MurDuPas(Jeu pred, Jeu cur)
{
    // Retourne (cote, k, mur) du pas pred -> cur, ou null si non adjacent.
    // TODO etudiant
    return null;   // a remplacer
}

var predEx56 = ((2, 1, 4, 3), (3, 4, 1, 2));
var curEx56 = ((2, 1, 4, 3), (3, 4, 2, 1));
Console.WriteLine($"Exercice 1 à compléter : mur_du_pas {PyJeu(predEx56)} -> {PyJeu(curEx56)}");

Exercice 1 à compléter : mur_du_pas ((2, 1, 4, 3), (3, 4, 1, 2)) -> ((2, 1, 4, 3), (3, 4, 2, 1))


## Exercice 5 — Chercher le contre-exemple au fait mesuré

La section 3.2 a **mesuré** (10 paires nommées + 300 paires quelconques) que le chemin le plus court est aussi le moins cher sous `W = {1: 1, 2: 2, 3: 3}`. Le fait est-il robuste à un autre barème ? Choisir un autre tarif (p.ex. `W2 = {1: 1, 2: 1, 3: 3}` ou `{1: 3, 2: 1, 3: 1}`, le sommet bon marché), refaire le balayage — pour chaque paire nommée et un échantillon de 300 paires, comparer `cout_chemin(construire_chemin)` et `cout_chemin(chemin_min_cout)` — et rapporter : un barème où un détour gagne existe-t-il ?

Indice : commencer par le barème sommet bon marché — que devient l'écart moyen ?

In [24]:
// Exercice 2 -- chercher le contre-exemple au fait mesure
// La section 3.2 a MESURE (10 paires nommees + 300 paires quelconques) que le chemin le plus court
// est aussi le moins cher sous W = {1: 1, 2: 2, 3: 3}. Est-ce robuste a un autre bareme ?
// 1. choisir un autre tarif, p.ex. W2 = {1: 1, 2: 1, 3: 3} ou {1: 3, 2: 1, 3: 1} (sommet bon marche !) ;
// 2. re-faire le balayage : pour chaque paire nommee et un echantillon de 300 paires,
//    comparer CoutChemin(ConstruireChemin) et CoutChemin(CheminMinCout) ;
// 3. rapporter : un bareme ou un detour gagne existe-t-il ?
// Indice : commencer par le bareme sommet bon marche -- que devient l'ecart moyen ?

var W2 = new Dictionary<int, int> { [1] = 1, [2] = 2, [3] = 3 };  // TODO etudiant : essayer d'autres baremes
Console.WriteLine($"Exercice 2 à compléter : balayage sous barème {PyDictInt(W2)}");

Exercice 2 à compléter : balayage sous barème {1: 1, 2: 2, 3: 3}


## Exercice 6 — Capstone : POULE -> RENVERSE, le parcours complet

Faire courir `parcours_complet` de la Poule au Renversement sous `W = {1: 1, 2: 2, 3: 3}`, puis passer le résultat à `verifier_parcours`. Questions de lecture : combien de pas, et répartis comment entre les deux joueurs ? Quels niveaux sont révisés — le trajet touche-t-il le sommet de qui que ce soit ? La facture par joueur : qui paie le plus, et pourquoi la géométrie l'impose-t-elle (cf 3.1) ?

In [25]:
// Exercice 3 -- capstone : POULE -> RENVERSE, le parcours complet
// Faire courir ParcoursComplet de la Poule au Renversement sous W = {1: 1, 2: 2, 3: 3},
// puis passer le resultat a VerifierParcours. Questions de lecture :
// 1. combien de pas, et repartis comment entre les deux joueurs ?
// 2. quels niveaux sont revises -- le trajet touche-t-il le sommet de qui que ce soit ?
// 3. la facture par joueur : qui paie le plus, et pourquoi la geometrie l'impose-t-elle (cf 3.1) ?

Console.WriteLine("Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification");

Exercice 3 à compléter : parcours POULE -> RENVERSE + vérification


## 9. Preuve formelle — le module `Abstraction/Basic.lean`, consommé aux lignes mesurées

Tout ce carnet a **mesuré** le prix des méta-actions : seuils de migration (marche 2), tarifs par niveau (section 7), et ce fait fragile du 3.2 — le plus court chemin est aussi le moins cher, *sur les paires testées*. Le lac `game_theory_lean` du dépôt porte un module qui pousse cette question un étage plus haut dans la formalisation : `Abstraction/Basic.lean` (grain #12204) certifie par le noyau Lean ce qui arrive quand la méta-action n'est plus *déplacer le jeu* mais **raffiner l'abstraction sous laquelle on joue** — et sa réponse est une réfutation. Le module est **volontairement sans Mathlib** (pattern `Swaps`, cf [GT-3b, section 10-bis](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb)) : tout s'y clôt par `decide`/`rfl` sur des littéraux entiers.

Comme en 10-bis de GT-3b, la consommation se fait **sans lancer `lake` en sous-processus** : la cellule suivante lit le fichier source depuis ce dépôt, extrait chaque déclaration **à sa ligne réelle**, et vérifie la propreté axiomatique sur les lignes de code seules (docstrings et commentaires retirés). Les citations de la section s'appuient sur ces numéros, pas sur une mémoire.

In [26]:
// === Preuve formelle : localiser Abstraction/Basic.lean, extraire, verifier la proprete ===
// Patron GT-3b (5-bis) et GT-6b (cellule 2) : lecture du source, jamais de lake en sous-processus.
using System;
using System.IO;
using System.Text.RegularExpressions;

string TrouverAbstractionLean()
{
    // Localise Abstraction/Basic.lean depuis le repertoire courant ou ses parents
    // (le notebook vit dans GameTheory/, le module dans game_theory_lean/).
    var rels = new[]
    {
        Path.Combine("game_theory_lean", "Abstraction", "Basic.lean"),
        Path.Combine("MyIA.AI.Notebooks", "GameTheory", "game_theory_lean", "Abstraction", "Basic.lean")
    };
    for (var baseDir = new DirectoryInfo(Directory.GetCurrentDirectory()); baseDir is not null; baseDir = baseDir.Parent)
        foreach (string rel in rels)
        {
            string cible = Path.Combine(baseDir.FullName, rel);
            if (File.Exists(cible)) return cible;
        }
    return null;
}

var DECL_RE = new Regex(@"^(theorem|lemma|def|noncomputable def|abbrev|structure|inductive|instance)\s+([A-Za-z_][A-Za-z0-9_']*)");

string StripLeanNoise(string text)
{
    // Retire docstrings /- -/ et commentaires -- : il ne reste que les lignes de code.
    text = Regex.Replace(text, @"/-.*?-/", " ", RegexOptions.Singleline);
    return Regex.Replace(text, @"--[^\n]*", " ");
}

var cheminModule = TrouverAbstractionLean();
if (cheminModule is null)
    Console.WriteLine("module Abstraction/Basic.lean introuvable -- cloner le depot complet pour lire les citations");
else
{
    string src62 = File.ReadAllText(cheminModule);
    // str.splitlines() de Python : on decoupe sur \n, on retire le \r eventuel, et on jette
    // l'entree vide finale (309 lignes dans le fichier, pas 310).
    var lignes62 = src62.Split('\n').Select(l => l.EndsWith("\r") ? l.Substring(0, l.Length - 1) : l).ToList();
    if (lignes62.Count > 0 && lignes62[lignes62.Count - 1].Length == 0) lignes62.RemoveAt(lignes62.Count - 1);
    string code62 = StripLeanNoise(src62);
    var decls62 = new List<(int L, string K, string N)>();
    for (int i = 0; i < lignes62.Count; i++)
    {
        var m62 = DECL_RE.Match(lignes62[i]);
        if (m62.Success) decls62.Add((i + 1, m62.Groups[1].Value, m62.Groups[2].Value));
    }
    int nThm62 = decls62.Count(d => d.K == "theorem" || d.K == "lemma");
    var base62 = new DirectoryInfo(Path.GetDirectoryName(cheminModule)).Parent.Parent.FullName;
    Console.WriteLine($"{Path.GetRelativePath(base62, cheminModule)} : {lignes62.Count} lignes, {decls62.Count} declarations dont {nThm62} theoremes");
    Console.WriteLine();
    var cles62 = new[] { "encadrement", "dette_nonneg", "selle_dette_nulle", "selle_blocQ", "selle_blocP", "raffinement_aggrave_la_dette", "courbe_dette" };
    Console.WriteLine("declarations portees par le sujet de ce carnet :");
    foreach (var (ligne, kind, nom) in decls62)
        if (cles62.Contains(nom))
            Console.WriteLine(Fi($"  L{ligne,4} [{kind,-8}] {nom}"));
    Console.WriteLine();
    Console.WriteLine("proprete axiomatique, comptee sur les lignes de code (prose retiree) :");
    foreach (string mot in new[] { "sorry", "sorryAx", "native_decide", "axiom" })
        Console.WriteLine($"  occurrences de '{mot}' : {Regex.Matches(code62, @"\b" + mot + @"\b").Count}");
}

game_theory_lean\Abstraction\Basic.lean : 309 lignes, 45 declarations dont 21 theoremes


declarations portees par le sujet de ce carnet :


  L 128 [theorem ] encadrement


  L 139 [theorem ] dette_nonneg


  L 145 [theorem ] selle_dette_nulle


  L 226 [theorem ] selle_blocQ


... (4 outputs left out)

  occurrences de 'sorry' : 0


  occurrences de 'sorryAx' : 0


  occurrences de 'native_decide' : 0


  occurrences de 'axiom' : 0


### Lecture du résultat : l'inventaire est compté, la dette axiomatique nulle

Le module est réel et mesuré : l'inventaire imprimé ci-dessus couvre tout le fichier — définitions du modèle, lois générales, contre-exemple — et les déclarations que ce carnet mobilise y sont citées à leurs lignes réelles : les lois générales `encadrement`, `dette_nonneg`, `selle_dette_nulle`, les selles de blocs `selle_blocQ` et `selle_blocP`, le contre-exemple-phare `raffinement_aggrave_la_dette` et son résumé numérique `courbe_dette`. La propreté se compte sur les **lignes de code seules** — l'en-tête du module parle *de* `sorry` en prose, c'est précisément ce que le retrait des docstrings neutralise : zéro `sorry`, zéro `sorryAx`, zéro `native_decide`, zéro `axiom` déclaré. L'en-tête certifie de plus que le seul axiome traversant les lemmes d'ordre généraux de `Int` est `propext`, dont aucun théorème concret ne dépend — tout se clôt par réduction du noyau (`decide`/`rfl`), exactement comme `Swaps` en 5-bis de GT-3b.

In [27]:
// === Preuve formelle : miroir C# du module, execute sur le contre-exemple certifie ===
// Miroirs fideles des definitions de Abstraction/Basic.lean : duel 2x2 a somme nulle sur les entiers.

int Gain(Tab m, bool i, bool j)
{
    // Gain de Ligne pour le profil (i, j) ; m = (a, b, c, d) ligne par ligne (L75).
    var (a, b, c, d) = (m.Item1, m.Item2, m.Item3, m.Item4);
    var table = new Dictionary<(bool, bool), int>
    {
        [(false, false)] = a, [(false, true)] = b, [(true, false)] = c, [(true, true)] = d
    };
    return table[(i, j)];
}

int VHaut(Tab m, bool j)
{
    // Meilleure reponse ligne contre j : max des deux lignes (L83).
    return Math.Max(Gain(m, false, j), Gain(m, true, j));
}

int VBas(Tab m, bool i)
{
    // Meilleure defense colonne contre i : min des deux colonnes (L86).
    return Math.Min(Gain(m, i, false), Gain(m, i, true));
}

int Dette(Tab m, (bool I, bool J) s)
{
    // Dette (exploitabilite) du profil s : vHaut - vBas (L93).
    return VHaut(m, s.J) - VBas(m, s.I);
}

bool EstSelle(Tab m, (bool I, bool J) s)
{
    // Selle pure locale : le gain atteint vHaut et vBas (L96).
    int g = Gain(m, s.I, s.J);
    return g == VHaut(m, s.J) && g == VBas(m, s.I);
}

Tab SommeBloc(List<Tab> duels)
{
    // Duel abstrait d'un bloc d'etats : SOMME des duels du bloc (L107).
    var res = (0, 0, 0, 0);
    foreach (var m in duels) res = (res.Item1 + m.Item1, res.Item2 + m.Item2, res.Item3 + m.Item3, res.Item4 + m.Item4);
    return res;
}

int DetteBloc(List<Tab> duels, (bool I, bool J) s)
{
    // Dette totale retransportee d'une strategie a travers un bloc (L112).
    return duels.Sum(m => Dette(m, s));
}

// Les trois etats du contre-exemple (L163-L169)
var d0 = (1, 0, 0, -1);
var d1 = (-2, 0, 0, 1);
var d2 = (2, 0, -1, 0);

var blocQ = SommeBloc(new List<Tab> { d0, d1, d2 });
var blocP = SommeBloc(new List<Tab> { d1, d2 });   // blocQ L181, blocP L184
Console.WriteLine($"blocQ = {PyTup(blocQ)} -- blocQ_valeur L202 | blocP = {PyTup(blocP)} -- blocP_valeur L205");

var sQ = (false, true);
var sP = (false, false);                           // stratBlocQ L187, stratBlocP L190
int detQ = DetteBloc(new List<Tab> { d0, d1, d2 }, sQ);                                  // dette_partitionQ L261
int detP = Dette(d0, (false, true)) + DetteBloc(new List<Tab> { d1, d2 }, sP);           // dette_partitionP L268
int detD = Dette(d0, (false, true)) + Dette(d1, (true, false)) + Dette(d2, (false, true)); // dette_discrete L274
Console.WriteLine($"dette retransportee, partition grossiere Q : {detQ}");
Console.WriteLine($"dette retransportee, partition raffinee  P : {detP}");
Console.WriteLine($"dette retransportee, partition discrete  D : {detD}");
Console.WriteLine($"selles d'existence verifiees par le miroir : blocQ {PyBool(EstSelle(blocQ, sQ))} (L226), blocP {PyBool(EstSelle(blocP, sP))} (L236) -- l'unicite est certifiee en L229/L239");
Console.WriteLine();
Console.WriteLine($"courbe de courbe_dette (L306), reproduite par le miroir : 0 < {detQ} < {detP}, discret = {detD}");
if (!(0 < detQ && detQ < detP && detD == 0))
    throw new InvalidOperationException("le miroir doit reproduire la courbe certifiee par le noyau");   // garde : equivalent de l'assert Python
Console.WriteLine("miroir et noyau disent la meme courbe : le contre-exemple est reproduce, chiffre par chiffre");

blocQ = (1, 0, -1, 0) -- blocQ_valeur L202 | blocP = (0, 0, -1, 1) -- blocP_valeur L205


dette retransportee, partition grossiere Q : 3


dette retransportee, partition raffinee  P : 4


dette retransportee, partition discrete  D : 0


selles d'existence verifiees par le miroir : blocQ True (L226), blocP True (L236) -- l'unicite est certifiee en L229/L239


courbe de courbe_dette (L306), reproduite par le miroir : 0 < 3 < 4, discret = 0


miroir et noyau disent la meme courbe : le contre-exemple est reproduce, chiffre par chiffre


### Lecture du résultat : le miroir exécuté dit 0 < 3 < 4, discret 0

Le miroir C# reproduit chiffre par chiffre la courbe certifiée : le duel abstrait du bloc grossier vaut `(1, 0, -1, 0)` et celui du bloc raffiné `(0, 0, -1, 1)` — exactement les valeurs closes par `rfl` (`blocQ_valeur`, `blocP_valeur`) — et la dette retransportée passe de **3** (partition grossière : la selle `(0,1)` du bloc unique coûte 0 à l'état 0, 3 à l'état 1, 0 à l'état 2) à **4** (partition raffinée : selle propre de {0} gratuite, selle `(0,0)` du bloc {1,2} coûtant 2 + 2), avant de retomber à **0** dans le discret. La garde finale échouerait si le miroir divergeait de la courbe du noyau : c'est une vérification, pas une décoration.

### Ce que le module certifie — et ce qu'il réfute

Le pont avec ce carnet est **méthodologique, et il faut le dire** : le module ne calcule pas dans le substrat ordinal (tables de rangs 1-4) des marches 1-3, mais dans des duels 2×2 à somme nulle sur les entiers — la dette qu'il certifie n'est pas un chiffre mesuré ici. Ce qu'il apporte tient en deux temps.

**D'abord une formalisation de la marche 1.** Les trois lois générales disent, dans le modèle du module, pourquoi la dynamique de meilleure réponse ne peut se reposer qu'à l'équilibre pur : le gain de tout profil est encadré entre `vBas` et `vHaut` (`encadrement`), sa dette — l'écart exploitable qui reste — est non négative (`dette_nonneg`), et une selle pure, exactement un point de repos de la dynamique, a dette nulle (`selle_dette_nulle`). C'est la version certifiée du croisement mesuré en marche 1 : convergence si et seulement si équilibre pur.

**Ensuite une réfutation — et elle contredit une intuition que ce carnet aurait pu être tenté de généraliser.** La méta-action du module est *raffiner l'abstraction du jeu* : découper les blocs d'états sous lesquels on joue une selle commune. Le théorème-phare (`raffinement_aggrave_la_dette`) certifie que sur ses trois duels, raffiner {{0,1,2}} en {{0},{1,2}} **aggrave strictement** la dette retransportée, de 3 à 4, quel que soit le choix des selles — elles sont uniques (`selle_blocQ_unique`, `selle_blocP_unique`) — avant que le discret ne retombe à 0 : plus fin n'est pas toujours moins cher. La non-croissance de la dette observée par le notebook GT-19 sur *ses* matrices n'est donc pas un théorème, c'est une propriété de ses données. La leçon retombe directement sur la section 3.2 : « le plus court chemin est aussi le moins cher » y est un **fait mesuré** sur des paires testées, pas une loi, et le module montre sur un cousin de cette intuition ce qu'un contre-exemple certifié fait à une généralisation prématurée. La frontière entre théorème (3.1, la décomposition du coût) et mesure (3.2) traverse tout ce carnet ; le module la garde vivante en rappelant qu'une intuition mesurée n'est pas une loi certifiée.

## Conclusion du parcours complet

Le critère d'intégration du chantier est tenu, bout en bout et sur un seul écran : **un jeu nommé** (le Dilemme), **un chemin non écrit à la main** (BFS + parents, vérifié), **des murs visibles** (chaque pas exhibe la table à égalité entre les deux jeux, ses deux faces confirmées), **des coûts lus** (deux révisions de sommet, 3 échelons chacune, payées séparément) -- et l'arrivée **porte un nom** (la Chasse au Cerf).

Deux résultats structurent le parcours au-delà de l'assemblage. Le **théorème des bornes** (3.1) : le coût du trajet minimal se décompose en `c_L·d_L + c_C·d_C` où `d_L`, `d_C` sont des distances de tables individuelles -- la facture est un invariant géométrique, hors de portée de tout marchandage sur le choix du chemin. Et le **fait mesuré** (3.2) : sous tarification par niveau, le plus court chemin est aussi le moins cher, sur toutes les paires testées -- l'intuition du détour économique ne survit pas à la mesure.

**Pour aller plus loin** : l'arbitrage *migrer ou rester* (à quel prix un agent accepte-t-il de payer ces 6 échelons ?) est le sujet de la Partie I (marche 2, sections 2-3 ci-dessus) ; la théorie des murs et des chambres, celui de [GT-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) ; la séparation constructeur / vérificateur, celle de [Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb).

## Conclusion générale : trois marches, puis un parcours

Les trois marches mesurent le passage de la strate 6 a la strate 7 sur un univers fini ou tout se verifie :

| Marche | Ce que fait l'agent | Chiffre cle |
|---|---|---|
| 1 -- jouer | subit les regles, converge ou cycle | 72 jeux injouables sur 576 ; convergence si et seulement si equilibre pur |
| 2 -- deplacer | paie des echelons pour reecrire sa table | 56 % -> 16 % -> 8 % -> 4 % de migrants quand le cout monte ; le Dilemme exactement indifferent a c=1 |
| 3 -- se coordonner sur les regles | joue le meta-jeu 4x4 | 106 jeux ou bouger est necessaire ; le Dilemme s'evade a (3,3) par equilibre symetrique ; 4 echecs de coordination dur |

Le resultat qui donne son sens au chantier : **changer les regles est une action ordinaire** -- elle a un prix, un seuil, des equilibres, et meme ses propres echecs de coordination. Le vocabulaire strategique de la strate 6 s'applique sans modification un etage plus haut ; c'est la definition meme d'un franchissement reussi.

*Suite du chantier (#12207)* : D1 (la grammaire des generateurs manipulables) et D3 (le chemin minimal certifie, cible Lean a terme via #12205) restent a ecrire sur le meme substrat.

## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| Jouer est une dynamique, pas un arbitrage ponctuel | `NE purs par jeu : {0: 72, 1: 432, 2: 72} sur 576` et `Dynamique BR depuis TG : converge 504`, `cycle 72` ; le croisement `{(0, 'cycle'): 72, (1, 'ok'): 432, (2, 'ok'): 72}` établit que converger équivaut à posséder un équilibre pur |
| Changer les règles a un prix, et le prix sélectionne | le sweep fait tomber la part de migrants de `56%` sur la première ligne (`c = 0`) à `16%`, `8%` puis `4%` sur les trois lignes suivantes, avec une distance optimale moyenne qui descend de `1.33` à `1.25` puis `1.00` |
| Le gain net des migrants n'est pas monotone en le coût | `+1.93` à `c = 0`, `+2.00` à `c = 1`, puis `+1.50` à `c = 2` et `+1.00` sur la dernière ligne : la population migrante se concentre d'abord sur les gains forts, ensuite le prix prélevé fait baisser la moyenne |
| Le Dilemme est exactement indifférent à `c = 1` | rang courant `2` ; à distance `1` on atteint le rang `3` (net `2`), à distance `2` le rang `4` (net `2`), à distance `3` le net retombe à `1` ; verdict affiché : à `c = 0`, `Ligne MIGRE a distance 2` ; à `c = 1` comme à `c = 2`, `Ligne RESTE` |
| Partir, c'est parfois fuir l'injouable | `Jeux injouables : 72`, `reparables en 1 swap : 48`, `autres : 24` ; à `c = 1`, `Migrants a c=1 : 96 jeux sur 576 dont 72 refugies` |
| Le mur est une égalité de rangs, pas une frontière de cases | à `lambda = 0.50`, `MUR : u_TG = u_BG -- codimension 1` ; `Murs simples distincts par joueur : 36` ; `Propriété mur <-> paire de chambres, vérifiée sur 24 tables x 3 niveaux : True` |
| Le méta-jeu a lui aussi ses équilibres | `Jeux avec au moins un meta-NE pur :` et `572 / 576` ; `aucun meta-NE ne reste sur place : 106` (bouger est alors nécessaire) contre `134` jeux où tous les équilibres restent en place |
| La coordination peut échouer, et l'échec se compte | `Jeux (avec meta-NE) dont AU MOINS UN equilibre est Pareto-optimal : 568` et `Jeux ou AUCUN equilibre n'est Pareto-optimal : 4` |
| Le chemin le plus court est aussi le moins cher | BFS et Dijkstra coïncident sur toutes les paires nommées (`écart 0`), et `Paires nommées où un détour battrait le plus court chemin : 0 / 10` |
| Le coût est géométrique : il ne se négocie pas | `Tarifs par côté : Ligne 1, Colonne 3` ; la ligne `PD -> CERF` du tableau porte d_L 1, d_C 1, longueur 2 et coût 4, quand la ligne `PD -> IDENTITE` porte d_L 3, d_C 4, longueur 7 et coût 15 |
| Le témoin se construit et se vérifie séparément | `VALIDE + MINIMAL : 2 pas = distance exhaustive d(G,H)` pour le chemin du constructeur, `INVALIDE : pas non élémentaire` pour un pas vers un jeu non adjacent |
| Le formel certifie la dette, il ne la calcule pas | `dette retransportee, partition grossiere Q : 3`, `dette retransportee, partition raffinee  P : 4`, `dette retransportee, partition discrete  D : 0`, et la courbe `0 < 3 < 4, discret = 0` reproduite par le miroir C# |

**Ce que ce carnet ajoute à la série** : il ne se contente pas de constater qu'un agent peut réécrire les règles — il en mesure le **prix**, le **seuil**, l'**équilibre** et l'**échec de coordination**, puis il referme la boucle par un parcours dont chaque pas est exhibé (jeu de départ, jeu d'arrivée, mur traversé, faces du mur, coût payé) et dont le témoin est vérifié par un vérificateur indépendant. Le versant formel (`game_theory_lean\Abstraction\Basic.lean`) est consommé à la ligne près : `selle_blocQ` (L226), `selle_blocP` (L236), `courbe_dette` (L306), avec une impureté axiomatique comptée à `0`.

## Vérifiez votre compréhension

**1. Le sweep fait tomber la part de migrants de `56%` sur sa première ligne à `4%` sur la dernière, et le gain net moyen des migrants *augmente* d'abord : `+1.93` à `c = 0`, `+2.00` à `c = 1`. Un coût plus élevé devrait au contraire réduire le gain de ceux qui migrent. Que se passe-t-il ?**

<details><summary>Réponse</summary>

Le gain net moyen est une moyenne **conditionnelle** : elle porte sur la population qui migre, et cette population change avec le prix. À `c = 0`, `56%` des jeux vues ont un agent qui migre, y compris des cas où l'écart de rang est faible : la moyenne est diluée (`+1.93`, distance optimale moyenne `1.33`). À `c = 1`, seuls `16%` migrent — le prix a éliminé les gains marginaux, et il reste ceux qui ont le plus à perdre au jeu courant : la moyenne monte (`+2.00`) malgré un coût ajouté. Ensuite le gain net redescend (`+1.50` à `c = 2`, puis `+1.00` sur la dernière ligne), non plus par sélection mais parce que le net est un net : le même gain brut, amputé du prix payé. En clair, le prix agit **deux fois** — il filtre les migrants, puis il prélève sur eux.

</details>

**2. Le tableau des tarifs par côté annonce un coût de 4 pour `PD -> CERF`, et le parcours complet annonce une `facture totale 6 (Ligne 3, Colonne 3)` pour le même trajet. Lequel des deux chiffres est le bon ?**

<details><summary>Réponse</summary>

Les deux, parce qu'ils ne tarifent pas la même chose. Le tableau de la section « tarifs par côté » facture des **déplacements de tables** : le coût y vaut `c_L·d_L + c_C·d_C`, avec `c_L` valant 1 et `c_C` valant 3 (`Tarifs par côté : Ligne 1, Colonne 3`), ce qui donne pour `PD -> CERF` d_L 1 et d_C 1, soit 1 + 3 = 4. Le barème du parcours complet facture des **niveaux** : `{1: 1, 2: 2, 3: 3}`, et les deux pas du parcours (un échange de niveaux `3 <-> 4` de chaque côté) traversent donc le niveau `3` à chaque fois, soit 3 + 3 = 6. C'est la même géométrie, deux modèles de prix : l'un compte les cases franchies, l'autre les niveaux traversés. Le carnet ne les mélange jamais, et c'est ce qui rend la remarque « le plus court est aussi le moins cher » (`écart 0` entre BFS et Dijkstra sur toutes les paires nommées) vérifiable plutôt que credite.

</details>

## Pour aller plus loin

- [README de la série GameTheory](README.md) : la place du carnet dans l'arc « les jeux comme objets » — le versant où l'agent achète le droit de réécrire les règles, avec un prix et un seuil.
- Les six exercices du carnet rejouent la mécanique plutôt que de la recopier : le sweep côté Colonne (exercice 1), la sélection d'équilibre déplaçable (exercice 2), le méta-jeu séquentiel (exercice 3), le mur du pas dans l'autre sens (exercice 4), la recherche du contre-exemple au fait mesuré (exercice 5), et le capstone `POULE -> RENVERSE` (exercice 6) qui rejoue le parcours complet sur une autre paire.
- Le module `game_theory_lean\Abstraction\Basic.lean` (`45 declarations dont 21 theoremes`) porte la version formelle du versant dette : `encadrement` (L128), `dette_nonneg` (L139), `selle_dette_nulle` (L145), `selle_blocQ` (L226), `selle_blocP` (L236), `raffinement_aggrave_la_dette` (L294), `courbe_dette` (L306). Le carnet en consomme l'inventaire et en rejoue la courbe par un miroir C#.
- Les carnets voisins prolongent la même marche : GT-3b pour la théorie des murs et des chambres, GT-20 pour la séparation constructeur / vérificateur des chemins, GT-21 pour les deux espèces de flèches sur le même espace.

***

[← GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) | [↑ README GameTheory](README.md) | [jumeau Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) · [jumeau Rust](GameTheory-03e-Meta-Actions-Tarifees-Rust.ipynb)

*GameTheory 3e -- Meta-Actions Tarifees, versant D4 du chantier « Les jeux comme objets » (#12207). Substrat, encodages et swaps herites de GT-3b ; jumeaux conceptuels GT-21 (les deux especes de fleches) et GT-20 (l'engagement comme protocole de coordination sur les regles).*